# Paper 5, W8 - single-head rebuilding controls for the ICLR rebuttal

**What this run is for.** The mock review's main RoPE criticism is that every "head" in §4.3, §4.5,
Fig 4d, Fig 5b and Appendix F is one head *index* spanning blocks 6-11, so we cannot say which
(block, head) does the rebuilding. W8 replaces the index groups with single (block, head) units and
re-runs every RoPE claim that rests on them, plus the other RoPE-side gaps we found.

**Naming.** The repo already has a `w7/` (the Sept reviewer-controls run behind Table 18 and E.2), so
this run is **W8** and writes to `./w8_output/`. Nothing in `w7/` is touched.

| Part | What | Data | Models |
|---|---|---|---|
| A | every single (block, head) rotation-masked in blocks 6-11 after the 0-5 window; SSDC at every block **and at the output of block 11**; paired image bootstrap; extra RPI seeds for the top heads; per-head output norms | repack, 1,000 | naver, DINOv3, small, reg1-gap, hybrid |
| B | axis attribution of single heads with row/col probes (the Table 14 / Fig 4d decomposition) + probe noise at the new readout | repack, 1,000 | naver, DINOv3, small, reg1-gap |
| C | top-1 for every single head after the window, every index group (restores reg1-gap paired SEs), block knockouts, intact-model and shuffled-rotation checks, and the hybrid's all-heads test | ILSVRC, 5,000 | naver, small, reg1-gap, hybrid |
| D | is index grouping arbitrary? best-head-per-block set, top-6 singles, and random one-head-per-block sets | both | naver, DINOv3, small, reg1-gap |
| E | single-head carriers (blocks 0-3, read at block 4) so the F.2 carrier claims stop using index groups | repack, 1,000 | naver, DINOv3, small, reg1-gap |
| F | recipe-matched **mixed** RoPE ViT-B: can one axis still be removed when every rotation mixes rows and columns? (answers "axial by construction") | both | mixed |
| G | Table 9 reproduction at block 4 plus the APE ridge-R² metric on RoPE (insurance against the probe-asymmetry comment) | repack, 1,000 | naver, DINOv3, small, reg1-gap |

**Coherence with W2-W7.** SSDC and probe parts use the same repack, image count, batch size, seed,
fp32, RPI hook, readout site and probe protocol as W5, and gate against the committed W5 numbers
(Cref, grouped heads, Table 9, Table 14). Accuracy parts use the gated ILSVRC split like W7 and gate
against W7's per-image results (Table 18). Images are decoded once per model and replayed from memory;
a gate proves the replay gives bit-identical SSDC and top-1 to the repo's streamed path.

**How to run.** Set `W8_PARTS` / `W8_MODELS` to split the job; every (part, model) saves on completion
and `W8_RESUME=1` (default) skips finished ones after a restart.

In [ ]:
import importlib, subprocess, sys
def ensure(p, m=None):
    try:
        importlib.import_module(m or p); return False
    except ImportError:
        print(f"installing {p} ..."); subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", p]); return True
for p in ["timm", "transformers", "datasets", "scipy", "pandas"]:
    ensure(p)
print("environment ready")

In [ ]:
import os, json, math, time, warnings, traceback, shutil, hashlib
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import timm
warnings.filterwarnings("ignore", category=UserWarning)
print("torch", torch.__version__, "| timm", timm.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info(); print(f"memory: {free/1e9:.1f} / {total/1e9:.1f} GB free")

In [ ]:
# config
REPO_PATH  = os.environ.get("W8_REPO", "")          # .../vit-sae-analysis/project_code/src ("" = search)
PARTS      = [p for p in os.environ.get("W8_PARTS", "A,B,C,D,E,F,G").split(",") if p]
MODELS_RUN = [m for m in os.environ.get("W8_MODELS", "naver,DINOv3,small,reg1_gap,rope_ape,mixed").split(",") if m]
RESUME     = os.environ.get("W8_RESUME", "1") == "1"
TEST_MODE  = os.environ.get("W8_TEST", "0") == "1"  # CPU smoke test on tiny random models, not for results
OUT_DIR    = Path(os.environ.get("W8_OUT", "w8_output"))

N_SSDC, N_ACC, BATCH_SIZE = 1000, 5000, 256
METRIC      = "manhattan"
TORCH_SEED  = 20260908
INPUT_SIZE  = 224
SSDC_DATASET = os.environ.get("W8_SSDC_DATASET", "benjamin-paine/imagenet-1k-256x256")  # every W2-W6 SSDC/probe number
ACC_DATASET  = os.environ.get("W8_ACC_DATASET",  "ILSVRC/imagenet-1k")                  # W7 accuracy (Table 18)

N_BLOCKS = 12
WINDOW   = list(range(0, 6))      # rotations off here, as in W3-W7
POST     = list(range(6, 12))     # where rebuilding happens
EARLY    = list(range(0, 4))      # part E: blocks that can affect the block-4 readout
FINAL    = 12                     # readout key: output of block 11 after the model's final LayerNorm

PROBE_LR, PROBE_PASSES, PROBE_BATCH, PROBE_TRAIN_FRAC = 5e-3, 20, 512, 0.8   # W2-W6 protocol, unchanged
PROBE_CHUNK, PROBE_DEVICE = 8192, None
NOISE_SEEDS, NOISE_BOOT = [0, 1, 2], 200                                       # W5 part B probe noise

BOOT_B, BOOT_SEED = 500, 8086       # paired image bootstrap, same resamples for every condition
RPI_SEEDS_EXTRA   = [12345, 777]    # two of W7's E1.2 seeds, for the top heads only
A_SEED_TOPN       = 8               # singles re-measured under the extra RPI seeds
B_TOPK_SINGLES    = 6               # top singles (by part A) that get the axis treatment
B_NAMED_GROUPS    = {"naver": [6], "DINOv3": [2, 9, 1], "reg1_gap": [7, 3], "small": [2, 0]}  # heads named in §4.3/§4.5
P_REF    = [7, 8, 9, 10, 11, FINAL]  # probe readouts for references and index groups (7-11 = Table 14)
P_SINGLE = [11, FINAL]               # probe readouts for single heads
C_INTACT_TOPN     = 3
D_RAND_SSDC, D_RAND_ACC, D_SEED = 48, 24, 31337
F_RAND_PLANES, F_RAND_SEED = 16, 5150
SHUFFLE_PERM_SEED = 7                # W7's shuffled-rotation permutation (perms[0])

ANCHOR_TOL_SSDC  = 1e-4   # vs committed W5 values (different session; fp32 drift is ~1e-6)
ANCHOR_TOL_PROBE = 0.005  # vs committed W5 probes (half a point; the 3-sigma noise bar is 1.2)
ANCHOR_TOL_ACC   = 0.001  # vs committed W7 top-1 (0.1 points = 5 of 5,000 images)
PERIMG_AGREE_MIN = 0.998  # per-image agreement with W7 where W7 kept per-image files
EQUIV_TOL        = 1e-9   # within-run equivalences
UNTOUCHED_TOL    = 1e-12  # readouts upstream of a masked block must not move at all

MODEL_SPECS = {
    "naver":    dict(name="vit_base_patch16_rope_224.naver_in1k",          classifier=True,  hybrid=False, mixed=False),
    "DINOv3":   dict(name="vit_base_patch16_dinov3.lvd1689m",              classifier=False, hybrid=False, mixed=False),
    "small":    dict(name="vit_small_patch16_rope_224.naver_in1k",         classifier=True,  hybrid=False, mixed=False),
    "reg1_gap": dict(name="vit_base_patch16_rope_reg1_gap_256.sbb_in1k",   classifier=True,  hybrid=False, mixed=False),
    "rope_ape": dict(name="vit_base_patch16_rope_ape_224.naver_in1k",      classifier=True,  hybrid=True,  mixed=False),
    "mixed":    dict(name="vit_base_patch16_rope_mixed_224.naver_in1k",    classifier=True,  hybrid=False, mixed=True),
}
# Which parts apply to which model (anything else is skipped with a note).
PART_MODELS = {
    "A": ["naver", "DINOv3", "small", "reg1_gap", "rope_ape"],
    "B": ["naver", "DINOv3", "small", "reg1_gap"],
    "C": ["naver", "small", "reg1_gap", "rope_ape"],
    "D": ["naver", "DINOv3", "small", "reg1_gap"],
    "E": ["naver", "DINOv3", "small", "reg1_gap"],
    "F": ["mixed"],
    "G": ["naver", "DINOv3", "small", "reg1_gap"],
}
# Selection rules, fixed before the run.
RULE_SINGLE = "largest drop in SSDC at the output of block 11 (readout 12) vs Cref; ties by (block, head)"
RULE_BEST_PER_BLOCK = "in each block 6-11, the head with the largest single-head drop at readout 12"
RULE_TOP6 = "the six single heads with the largest drop at readout 12, any blocks"

if TEST_MODE:
    N_SSDC, N_ACC, BATCH_SIZE = 24, 32, 8
    PROBE_PASSES, NOISE_BOOT, BOOT_B = 2, 10, 20
    D_RAND_SSDC, D_RAND_ACC = 4, 2
    OUT_DIR = Path(os.environ.get("W8_OUT", "w8_test_output"))
OUT_DIR.mkdir(exist_ok=True, parents=True)

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H-%M-%SZ")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PDEV = PROBE_DEVICE or DEVICE
GATES = []
GATE_FILE = OUT_DIR / f"w8_gates_session_{RUN_STAMP}.json"
def gate(name, scope, passed, detail="", skip=False):
    status = "SKIP" if skip else ("PASS" if passed else "FAIL")
    GATES.append({"gate": name, "scope": scope, "status": status, "detail": detail, "session": RUN_STAMP})
    print(f"  [{status}] {scope}: {name}" + (f": {detail}" if detail else ""), flush=True)
    GATE_FILE.write_text(json.dumps(GATES, indent=1))   # kept per session, so a resumed run still reports every gate
    return passed
print(f"device {DEVICE} | probes on {PDEV} | fp32 | parts {PARTS} | models {MODELS_RUN}")
print(f"SSDC/probes: {SSDC_DATASET} x {N_SSDC} | accuracy: {ACC_DATASET} x {N_ACC} | batch {BATCH_SIZE}")
print(f"TF32 matmul {torch.backends.cuda.matmul.allow_tf32} | cudnn TF32 {torch.backends.cudnn.allow_tf32} (left at defaults)")
print(f"output {OUT_DIR.resolve()} | resume {RESUME} | test mode {TEST_MODE} | run {RUN_STAMP}")

In [ ]:
def resolve_repo():
    cands = []
    if REPO_PATH:
        cands.append(Path(REPO_PATH).expanduser())
    here = Path.cwd()
    for base in [here, *list(here.parents)[:4]]:
        cands += [base / "vit-sae-analysis" / "project_code" / "src", base / "project_code" / "src", base / "src", base]
    for c in cands:
        c = c.resolve()
        if (c / "metrics").exists() and (c / "main").exists() and (c / "experiments").exists():
            return c
    raise FileNotFoundError("No checkout with metrics/, main/, experiments/. Set W8_REPO. Tried: " +
                            ", ".join(str(c) for c in cands[:8]))

repo = resolve_repo()
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
print("repo root:", repo)

from metrics.ssdc import (evaluate_ssdc, spatial_similarity_distance_correlation, SSDCAccumulator,
                          _register_ssdc_accumulator_hooks, pairwise_cosine_similarity)
from main.load_models import get_vit_blocks, get_block_attention, get_patch_embed_conv, load_rope
from main.model import predict
from main.prep_data import prep_data
from experiments.common import load_imagenet
try:
    from metrics.position_probe import (evaluate_coordinate_probe, split_image_indices,
                                        DEFAULT_COORDINATE_PROBE_ALPHA_GRID)
    RIDGE_OK = True
except Exception as ex:
    RIDGE_OK = False
    print("ridge probe import failed (part G/F ridge columns will be skipped):", repr(ex))

from scipy.spatial.distance import cdist as _cdist
from scipy.stats import rankdata, spearmanr
_G = 14
_c = np.stack(np.meshgrid(np.arange(_G), np.arange(_G), indexing="ij"), -1).reshape(-1, 2)
_D = _cdist(_c, _c, metric="cityblock")
_a = spatial_similarity_distance_correlation(-_D, grid_size=_G, metric="manhattan")
_b = spatial_similarity_distance_correlation(_D, grid_size=_G, metric="manhattan")
_r = spatial_similarity_distance_correlation(np.random.default_rng(0).normal(size=(_G * _G,) * 2),
                                             grid_size=_G, metric="manhattan")
gate("imported metric satisfies its unit test", "repo-metric",
     _a > 0.999 and _b < -0.999 and abs(_r) < 0.1, f"{_a:+.3f}/{_b:+.3f}/{_r:+.3f}")

In [ ]:
# datasets: explicit ids, no silent fallback
if TEST_MODE:
    from PIL import Image as _PIL
    class _FakeStream:
        def __init__(self, n, seed):
            self.n, self.seed = n, seed
        def __iter__(self):
            rng = np.random.default_rng(self.seed)
            for i in range(self.n):
                arr = rng.integers(0, 256, size=(256, 256, 3), dtype=np.uint8)
                yield {"image": _PIL.fromarray(arr), "label": int(i % 10)}
    DS_SSDC = _FakeStream(400, 1); DS_ACC = _FakeStream(400, 2)
    ACC_IS_ILSVRC = True
    print("TEST MODE: random images stand in for both datasets")
else:
    DS_SSDC = load_imagenet(split="validation", streaming=True, dataset_id=SSDC_DATASET, fallback_id=None)
    DS_ACC, ACC_IS_ILSVRC = None, False
    if any(p in PARTS for p in ("C", "D", "F")):
        try:
            DS_ACC = load_imagenet(split="validation", streaming=True, dataset_id=ACC_DATASET, fallback_id=None)
            ACC_IS_ILSVRC = ACC_DATASET == "ILSVRC/imagenet-1k"
        except Exception as ex:
            print(f"\n!! could not load {ACC_DATASET}: {type(ex).__name__}: {ex}")
            print("!! accuracy parts will not run. Fix the HF token (W7 read this split) or set "
                  "W8_ACC_DATASET to the repack; with the repack the W7 accuracy anchors cannot be checked.")
    gate("SSDC dataset is the repack every W2-W6 SSDC/probe number used", "data",
         SSDC_DATASET == "benjamin-paine/imagenet-1k-256x256", SSDC_DATASET)
    gate("accuracy dataset is the ILSVRC split W7 used", "data", DS_ACC is not None and ACC_IS_ILSVRC,
         ACC_DATASET if DS_ACC is not None else "not loaded")

In [ ]:
# committed anchors (read from the repo's result files, never retyped)
RES_ROOT = repo.parents[1] / "results" / "runs" / "imagenet1k_val" / "rope_interventions"
def _first(pattern):
    hits = sorted(RES_ROOT.glob(pattern)); return hits[0] if hits else None
W5_FILES = {nm: _first(f"w5/w5_partA_{nm}_*.json") for nm in ["naver", "DINOv3", "small", "reg1_gap", "rope_ape"]}
W7_FILES = {"naver": RES_ROOT / "w7/w7rope_R12_naver_2026-09-24T17-54-52Z.json",
            "small": RES_ROOT / "w7/w7rope_R12_small_2026-09-24T22-49-53Z.json",
            "reg1_gap": RES_ROOT / "w7/w7rope_R12_reg1_gap_MERGED.json"}
W7_PERIMG = {nm: RES_ROOT / f"w7/perimage/R12_{nm}_correct.npz" for nm in W7_FILES}
# A few values typed in by hand, only to prove the right files were found.
HEADLINE = {"naver": (0.329663, 0.469422), "DINOv3": (0.442640, 0.948654), "small": (0.244217, 0.371434),
            "reg1_gap": (0.244575, 0.689023), "rope_ape": (0.263854, 0.589777)}   # (Cref b11, baseline b4)
HEADLINE_ACC = {"naver": (0.7052, 6, 0.6622), "small": (0.4868, 2, 0.4106), "reg1_gap": (0.5628, 7, 0.5054)}

ANCH, ACC_ANCH, W7_IMG = {}, {}, {}
for nm, f in W5_FILES.items():
    if f is None:
        gate("committed W5 file found", nm, False, "missing"); continue
    e = json.load(open(f)); c = e["conditions"]; nh = e["n_heads"]
    ANCH[nm] = {
        "file": f.name, "n_heads": nh, "row_planes": e["row_planes"], "col_planes": e["col_planes"],
        "baseline": c["baseline"]["ssdc"], "floor": c["floor"]["ssdc"], "Cref": c["Cref_prefix5"]["ssdc"],
        "Dref_row": c["Dref_axisrow5"]["ssdc"], "Dref_col": c["Dref_axiscol5"]["ssdc"],
        "grp_b11": {h: c[f"C_head{h:02d}"]["ssdc"][11] for h in range(nh)},
        "carrier_b4": {h: c[f"A_head{h:02d}"]["ssdc"][4] for h in range(nh)},
        "stageA": [d["head"] for d in e["stageA_ranking"]], "stageC": [d["head"] for d in e["stageC_ranking"]],
        "probe4": {k: c[k]["probes"].get("4") for k in ("baseline", "axis_row", "axis_col", "floor")},
        "ssdc4": {k: c[k]["ssdc"][4] for k in ("baseline", "axis_row", "axis_col", "floor")},
        "base_post": {int(b): c["baseline"]["probes"][b] for b in c["baseline"]["probes"] if int(b) >= 7},
        "Dref_post": {ax: {int(b): c[f"Dref_axis{ax}5"]["probes"][b] for b in c[f"Dref_axis{ax}5"]["probes"]}
                      for ax in ("row", "col")},
        "D_post": {},
    }
    for k in c:
        if k.startswith("D_row_head") or k.startswith("D_col_head"):
            ANCH[nm]["D_post"].setdefault(int(k[-2:]), {})[k[2:5]] = {int(b): v for b, v in c[k]["probes"].items()}
    cb11, b4 = HEADLINE[nm]
    gate("committed W5 file matches its headline values", nm,
         abs(ANCH[nm]["Cref"][11] - cb11) < 1e-6 and abs(ANCH[nm]["baseline"][4] - b4) < 1e-6, f.name)
for nm, f in W7_FILES.items():
    if not f.exists():
        gate("committed W7 file found", nm, False, str(f)); continue
    e = json.load(open(f)); c = e["conditions"]
    ACC_ANCH[nm] = {k: v["acc"] for k, v in c.items() if "acc" in v}
    ACC_ANCH[nm]["_top"] = e["top_rebuilder"]
    w, top, wt = HEADLINE_ACC[nm]
    gate("committed W7 file matches its headline values", nm,
         abs(ACC_ANCH[nm]["window_identity"] - w) < 1e-4 and e["top_rebuilder"] == top
         and abs(ACC_ANCH[nm][f"win_h{top}"] - wt) < 1e-4, f.name)
    if W7_PERIMG[nm].exists():
        z = np.load(W7_PERIMG[nm]); W7_IMG[nm] = {k: z[k] for k in z.files}
print("anchors:", {k: v["file"] for k, v in ANCH.items()})
print("W7 per-image arms:", {k: len(v) for k, v in W7_IMG.items()})

In [ ]:
# rotary helpers: from W5/W7, plus the two W8 additions at the end
def find_rope(model):
    for n, m in model.named_modules():
        if callable(getattr(m, "get_embed", None)): return n, m
    return None, None

def attn_attrs(model, source):
    rh, src, nh = False, "default(interleaved)", None
    for blk in get_vit_blocks(model, source):
        a = get_block_attention(blk, source)
        for at in ("rotate_half", "half", "rope_rotate_half"):
            v = getattr(a, at, None)
            if isinstance(v, bool): rh, src = v, at; break
        nh = getattr(a, "num_heads", None); break
    return rh, src, nh

def plane_columns(p, n, rh): return [p, p + n] if rh else [2 * p, 2 * p + 1]

def derive_axis(emb, n_planes, rh, H, W):
    sin, cos = emb.detach().float().chunk(2, -1); rows = []; dec = True
    for p in range(n_planes):
        c = plane_columns(p, n_planes, rh)[0]
        z = torch.complex(cos[:, c], sin[:, c]).reshape(H, W)
        ra = (z - z.mean(dim=1, keepdim=True)).abs().mean().item()
        rb = (z - z.mean(dim=0, keepdim=True)).abs().mean().item()
        ax = "row" if ra < rb else "col"; lo, hi = sorted([ra, rb]); r = lo / hi if hi > 1e-12 else 1.0
        if r > 0.10: dec = False
        rows.append({"plane": p, "axis": ax, "ratio": r})
    return rows, [r["plane"] for r in rows if r["axis"] == "row"], [r["plane"] for r in rows if r["axis"] == "col"], dec

def mask_planes(emb, kill, n, rh):
    if not kill: return emb
    s, c = emb.chunk(2, -1); s, c = s.clone(), c.clone()
    for p in kill:
        for col in plane_columns(p, n, rh): s[..., col] = 0.0; c[..., col] = 1.0
    return torch.cat([s, c], -1)

def identity_like(emb):
    s, c = emb.chunk(2, -1); return torch.cat([torch.zeros_like(s), torch.ones_like(c)], -1)
IDEN = identity_like

def per_head(heads, inner, n_heads):
    hs = sorted(set(heads))
    def tf(emb):
        if emb.dim() != 2:
            raise RuntimeError(f"per_head expected 2-D rope, got {tuple(emb.shape)}: stacked overrides")
        out = emb.unsqueeze(0).repeat(n_heads, 1, 1)
        if hs:
            t = inner(emb)
            for h in hs: out[h] = t
        return out
    return tf

def shuffled_like(perm):
    def tf(emb):
        if emb.dim() != 2: raise RuntimeError(f"shuffle expects 2-D rope, got {tuple(emb.shape)}")
        return emb[perm]
    return tf

class RopeGlobal:
    def __init__(s, model, tf): s.name, s.mod = find_rope(model); s.tf = tf; s.handles = []
    def __enter__(s):
        if s.mod is None: raise RuntimeError("no get_embed module")
        s.orig = s.mod.get_embed; o, t = s.orig, s.tf
        s.mod.get_embed = (lambda _o=o, _t=t: (lambda *a, **k: _t(_o(*a, **k))))()
        s.handles = [1]; return s
    def __exit__(s, *e): s.mod.get_embed = s.orig; s.handles = []

class RopeComposite:
    def __init__(s, model, source, segs):
        s.blocks = get_vit_blocks(model, source); s.source = source
        s.segs = [(list(b), t) for b, t in segs]; seen = set()
        for b, _ in s.segs:
            if seen & set(b): raise ValueError(f"overlapping segments: {sorted(seen & set(b))}")
            seen |= set(b)
        s.handles = []
    def __enter__(s):
        def mk(tf):
            def pre(mod, args, kw):
                r = kw.get("rope", None)
                if r is not None: kw = dict(kw); kw["rope"] = tf(r)
                return args, kw
            return pre
        for blocks, tf in s.segs:
            for i in blocks:
                s.handles.append(get_block_attention(s.blocks[i], s.source)
                                 .register_forward_pre_hook(mk(tf), with_kwargs=True))
        return s
    def __exit__(s, *e):
        for h in s.handles: h.remove()
        s.handles = []

class APEZero:
    def __init__(s, model): s.model = model; s.handles = []
    def __enter__(s):
        cands = [n for n, _ in s.model.named_parameters() if n.endswith("pos_embed") or n.endswith("position_embeddings")]
        if not cands: raise RuntimeError("no positional-embedding parameter")
        s.name = sorted(cands, key=len)[0]; parts = s.name.split("."); m = s.model
        for p in parts[:-1]: m = getattr(m, p)
        s.parent, s.attr = m, parts[-1]
        pe = getattr(s.parent, s.attr); s.saved = pe.detach().clone()
        with torch.no_grad(): pe.zero_()
        s.handles = [1]; return s
    def __exit__(s, *e):
        with torch.no_grad(): getattr(s.parent, s.attr).copy_(s.saved)
        s.handles = []

class Both:
    def __init__(s, model, tf): s.a = RopeGlobal(model, tf); s.b = APEZero(model); s.handles = []
    def __enter__(s): s.a.__enter__(); s.b.__enter__(); s.handles = [1, 1]; return s
    def __exit__(s, *e): s.b.__exit__(); s.a.__exit__(); s.handles = []

class NullCtx:
    handles = []
    def __enter__(s): return s
    def __exit__(s, *e): return False

# W8 addition 1: window plus any set of single (block, head) units
def win_plus(model, source, heads_by_block, n_heads, window_tf=IDEN, inner=IDEN, window=None):
    """Rotations off in the window (window_tf), plus `inner` on the listed heads. heads_by_block: {block: [heads]},
    blocks outside the window."""
    segs = [(list(WINDOW if window is None else window), window_tf)] if (window is None or window) else []
    for k in sorted(heads_by_block):
        hs = sorted(set(heads_by_block[k]))
        if hs: segs.append(([k], per_head(hs, inner, n_heads)))
    return RopeComposite(model, source, segs)

def heads_only(model, source, heads_by_block, n_heads, inner=IDEN):
    """No window: just the listed (block, head) units (part C intact checks, part E carriers)."""
    return RopeComposite(model, source, [([k], per_head(sorted(set(hs)), inner, n_heads))
                                         for k, hs in sorted(heads_by_block.items()) if hs])

# W8 addition 2: mixed-RoPE frequency masking (part F)
class MixedFreqMask:
    """Zero chosen coordinate components of the learned mixed-RoPE frequencies in place (restored on exit). comps:
    subset of {0, 1}; depths, planes: lists or None for all."""
    def __init__(s, rope, comps, depths=None, planes=None):
        s.rope, s.comps, s.depths, s.planes = rope, list(comps), depths, planes; s.handles = []
    def __enter__(s):
        f = s.rope.freqs; s.saved = f.detach().clone()
        m = torch.zeros_like(f, dtype=torch.bool)
        ds = range(f.shape[1]) if s.depths is None else s.depths
        ps = range(f.shape[3]) if s.planes is None else s.planes
        for c in s.comps:
            for d in ds:
                for p in ps: m[c, d, :, p] = True
        with torch.no_grad(): f[m] = 0.0
        s.handles = [1]; return s
    def __exit__(s, *e):
        with torch.no_grad(): s.rope.freqs.copy_(s.saved)
        s.handles = []

In [ ]:
# algebra gates (as in W5/W7) plus the W8 constructions
def _rot_il(x):
    x = x.unflatten(-1, (-1, 2)); return torch.stack([-x[..., 1], x[..., 0]], -1).flatten(-2)
def _rot_h(x):
    a, b = x.chunk(2, -1); return torch.cat([-b, a], -1)
def _apply(x, e, half=False):
    s, c = e.chunk(2, -1); return x * c + (_rot_h(x) if half else _rot_il(x)) * s

for lname, rh in [("interleaved", False), ("rotate_half", True)]:
    npl, dim, H = 32, 64, 12
    e = torch.randn(196, 2 * dim); x = torch.randn(2, H, 196, dim)
    gate(f"empty plane mask bit-identical [{lname}]", "algebra", torch.equal(mask_planes(e, [], npl, rh), e))
    gate(f"full plane mask equals identity [{lname}]", "algebra",
         torch.allclose(mask_planes(e, list(range(npl)), npl, rh), identity_like(e)))
    m = mask_planes(e, [3], npl, rh)
    touched = torch.nonzero((m - e).abs().sum(0) > 0).flatten().tolist()
    sc = plane_columns(3, npl, rh); exp = sorted(set(sc + [c + dim for c in sc]))
    gate(f"plane 3 touches only its own columns [{lname}]", "algebra", touched == exp, f"{touched} vs {exp}")
    full = _apply(x, e, rh); iden = _apply(x, identity_like(e), rh)
    gate(f"mask NO heads equals unmodified rope [{lname}]", "per-head",
         torch.allclose(_apply(x, per_head([], identity_like, H)(e), rh), full, atol=1e-6))
    gate(f"mask ALL heads equals global identity [{lname}]", "per-head",
         torch.allclose(_apply(x, per_head(range(H), identity_like, H)(e), rh), iden, atol=1e-6))
    one = _apply(x, per_head([3], identity_like, H)(e), rh); others = [h for h in range(H) if h != 3]
    gate(f"mask head 3 leaves others untouched [{lname}]", "per-head",
         torch.allclose(one[:, others], full[:, others], atol=1e-6))
    gate(f"mask head 3 changes head 3 [{lname}]", "per-head", not torch.allclose(one[:, 3], full[:, 3]))

class _Stub:
    def __init__(s, n=12): s.blocks = [nn.Identity() for _ in range(n)]
try:
    RopeComposite(_Stub(), "timm", [(range(0, 6), IDEN), (range(5, 12), IDEN)]); ov = False
except ValueError as ex:
    ov = "overlapping" in str(ex)
gate("composite REJECTS overlapping segments", "composite", ov)
try:
    _wp = win_plus(_Stub(), "timm", {k: [3] for k in POST}, 12)
    dj = [b for b, _ in _wp.segs] == [WINDOW] + [[k] for k in POST]
except Exception:
    dj = False
gate("win_plus builds the window plus one segment per listed block", "composite", dj)
try:
    win_plus(_Stub(), "timm", {5: [0]}, 12); bad = False
except ValueError:
    bad = True
gate("win_plus REJECTS a single head inside the window", "composite", bad)

In [ ]:
# image cache: decode once per model, replay through a real DataLoader
# the replay loader draws the same int64 from the global RNG as prep_data's loader, so predict()'s
# per-batch RPI permutations come out identical (checked by the setup gates)
class _CachedBatches(torch.utils.data.IterableDataset):
    def __init__(self, batches): super().__init__(); self.batches = batches
    def __iter__(self):
        for b in self.batches: yield b

def cached_loader(cache):
    return torch.utils.data.DataLoader(_CachedBatches(cache), batch_size=None, num_workers=0,
                                       pin_memory=torch.cuda.is_available(), shuffle=False)

def build_cache(dataset, processor, source, n):
    t = time.time()
    dl = prep_data(dataset, processor, source, corruption_type=None, number_images=n,
                   batch_size=BATCH_SIZE, half=False, num_workers=0)
    out = [(im.clone(), torch.as_tensor(lb).clone()) for im, lb in dl]
    got = sum(int(lb.numel()) for _, lb in out)
    print(f"    cached {got} images in {time.time() - t:.0f}s")
    return out

def cache_digest(cache):
    h = hashlib.sha1()
    for im, lb in cache[:2]:
        h.update(im[:4].numpy().tobytes()); h.update(lb[:4].numpy().tobytes())
    return h.hexdigest()[:12]

In [ ]:
# measurement
def final_norm_module(model):
    n = getattr(model, "norm", None)
    if n is not None and not isinstance(n, nn.Identity): return n, "norm"
    f = getattr(model, "fc_norm", None)
    if f is not None and not isinstance(f, nn.Identity): return f, "fc_norm"
    return nn.Identity(), "identity"

def neg_dist_ranks(T):
    G = int(round(T ** 0.5)); assert G * G == T
    c = np.stack(np.meshgrid(np.arange(G), np.arange(G), indexing="ij"), -1).reshape(-1, 2)
    D = _cdist(c, c, metric="cityblock"); iu = np.triu_indices(T, k=1)
    return rankdata(-D[iu]), iu

def gpu_spearman_rows(M, rk_std, iu_t):
    """Spearman of each row of M[B, T*T] (upper triangle) with the fixed negative-distance ranks; ordinal ranks, as
    in E1.2."""
    T = int(round(M.shape[1] ** 0.5))
    v = M.reshape(-1, T, T)[:, iu_t[0], iu_t[1]]
    r = v.argsort(1).argsort(1).float()
    r = r - r.mean(1, keepdim=True)
    return ((r * rk_std).sum(1) / (r.norm(dim=1) * rk_std.norm())).cpu().numpy()

def measure(M, ctx, cache_key="ssdc", seed=TORCH_SEED, boot_readouts=(), capture=(), headnorm=False):
    """One RPI sweep over the cached images under `ctx`: SSDC at every block input and at FINAL, with optional
    bootstrap SSDC, captured activations and per-head output norms."""
    model, source, n_prefix, nh = M["model"], M["source"], M["n_prefix"], M["n_heads"]
    cache = M["cache"][cache_key]
    blocks = get_vit_blocks(model, source)
    acc = SSDCAccumulator(n_prefix=n_prefix, spatial_metric=METRIC)
    handles = _register_ssdc_accumulator_hooks(model, source, acc)        # the repo's own readout
    fnorm = M["final_norm"]
    st = {"next": 0, "cur": 0}
    Wb = M["boot_w"] if boot_readouts else None
    boot = {r: None for r in boot_readouts}
    cap = {r: [] for r in capture}
    hn = torch.zeros(N_BLOCKS, nh, dtype=torch.float64) if headnorm else None
    hn_n = [0]

    def on_tokens(r, tok):
        if r in boot:
            t = tok[:, n_prefix:, :].float()
            S = pairwise_cosine_similarity(t).reshape(t.shape[0], -1)
            o = st["cur"]; contrib = Wb[:, o:o + t.shape[0]] @ S
            boot[r] = contrib if boot[r] is None else boot[r] + contrib
        if r in cap:
            cap[r].append(tok[:, n_prefix:, :].detach().float().cpu())

    extra = []
    extra.append(model.register_forward_pre_hook(
        lambda m, args: st.update(cur=st["next"], next=st["next"] + int(args[0].shape[0]))))
    for i in range(N_BLOCKS):
        if i in boot or i in cap:
            extra.append(get_block_attention(blocks[i], source).register_forward_hook(
                lambda m, inp, out, i=i: on_tokens(i, inp[0].detach())))
    def final_hook(m, inp, out):
        t = fnorm(out.detach())
        acc.add(FINAL, t)
        on_tokens(FINAL, t)
    extra.append(blocks[-1].register_forward_hook(final_hook))
    if headnorm:
        for i, blk in enumerate(blocks):
            proj = get_block_attention(blk, source).proj
            def pre(m, args, i=i):
                x = args[0].detach()[:, n_prefix:, :].float()
                B_, T_, C_ = x.shape; hd = C_ // nh
                Wo = m.weight.detach().float()
                for h in range(nh):
                    y = x[..., h * hd:(h + 1) * hd] @ Wo[:, h * hd:(h + 1) * hd].T
                    hn[i, h] += y.norm(dim=-1).mean(-1).sum().item()
                if i == 0: hn_n[0] += B_
            extra.append(proj.register_forward_pre_hook(pre))
    t0 = time.time()
    torch.manual_seed(seed); np.random.seed(seed % (2 ** 31))
    try:
        with ctx:
            hooks = len(getattr(ctx, "handles", []) or [])
            predict(model, cached_loader(cache), source, RPI=True, half=False)
    finally:
        for h in handles + extra: h.remove()
    ssdc = [acc.ssdc(i, remove_prefix=True) for i in range(N_BLOCKS)] + [acc.ssdc(FINAL, remove_prefix=True)]
    res = {"ssdc": [float(v) for v in ssdc], "hooks": hooks, "n_images": int(st["next"])}
    if boot_readouts:
        res["boot"] = {r: gpu_spearman_rows(boot[r], M["rk_std"], M["iu_t"]) for r in boot_readouts}
    if capture:
        res["acts"] = {r: torch.cat(cap[r], 0) for r in capture}
    if headnorm:
        res["head_norm"] = (hn / max(hn_n[0], 1)).tolist()
    res["runtime_s"] = round(time.time() - t0, 1)
    return res

def accuracy(M, ctx):
    """predict_paired from W6/W7 over the cached ILSVRC images: mean top-1 and per-image correctness."""
    model, cache = M["model"], M["cache"]["acc"]
    model.eval(); correct = []
    torch.manual_seed(TORCH_SEED)
    with ctx:
        hooks = len(getattr(ctx, "handles", []) or [])
        with torch.inference_mode():
            for images, labels in cached_loader(cache):
                pred = model(images.to(DEVICE)).argmax(-1).cpu()
                correct.append(pred == torch.as_tensor(labels).cpu())
    c = torch.cat(correct)
    return float(c.float().mean()), c.numpy().astype(np.uint8), hooks

# probes: W5's per-block-seeded protocol, unchanged
def _fit(Xtr, ytr, Xte, yte, nc, seed):
    torch.manual_seed(seed)
    head = nn.Linear(Xtr.shape[1], nc).to(Xtr.device)
    opt = torch.optim.AdamW(head.parameters(), lr=PROBE_LR)
    n = Xtr.shape[0]; peak = 0.0; best = None
    for _ in range(PROBE_PASSES):
        perm = torch.randperm(n, device=Xtr.device); head.train()
        for s0 in range(0, n, PROBE_BATCH):
            b = perm[s0:s0 + PROBE_BATCH]
            opt.zero_grad(); F.cross_entropy(head(Xtr[b]), ytr[b]).backward(); opt.step()
        head.eval(); correct = 0; preds = []
        with torch.no_grad():
            for s0 in range(0, Xte.shape[0], PROBE_CHUNK):
                p = head(Xte[s0:s0 + PROBE_CHUNK]).argmax(-1); preds.append(p)
                correct += (p == yte[s0:s0 + PROBE_CHUNK]).sum().item()
        acc_ = correct / Xte.shape[0]
        if acc_ > peak: peak = acc_; best = torch.cat(preds)
    del head, opt
    return float(peak), best

def _split(n_images, T, seed=TORCH_SEED):
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(n_images, generator=g); n_tr = int(round(n_images * PROBE_TRAIN_FRAC))
    rows = lambda ids: (ids.unsqueeze(1) * T + torch.arange(T).unsqueeze(0)).reshape(-1)
    return rows(idx[:n_tr]), rows(idx[n_tr:]), idx[n_tr:]

def probe_block(acts, H, W, block, heads=("exact", "row", "col"), noise=False):
    n_images, T, D = acts.shape
    feats = acts.reshape(-1, D)
    tr, te, te_img = _split(n_images, T)
    Xtr = feats[tr].to(PDEV); Xte = feats[te].to(PDEV)
    mu = Xtr.mean(0, keepdim=True); sd = Xtr.std(0, keepdim=True).clamp_min(1e-6)
    Xtr = (Xtr - mu) / sd; Xte = (Xte - mu) / sd
    slot = torch.arange(T); out = {}
    labels = {"exact": (slot, T), "row": (slot // W, H), "col": (slot % W, W)}
    for name in heads:
        lab, nc = labels[name]; y = lab.repeat(n_images)
        acc_, preds = _fit(Xtr, y[tr].to(PDEV), Xte, y[te].to(PDEV), nc, TORCH_SEED + block)
        out[name] = acc_
        if noise:
            corr = (preds == y[te].to(PDEV)).float()
            g = np.random.default_rng(TORCH_SEED + block)
            n_te = len(te_img); per = corr.reshape(n_te, T).mean(1).cpu().numpy()
            bt = [per[g.integers(0, n_te, n_te)].mean() for _ in range(NOISE_BOOT)]
            seeds = [_fit(Xtr, y[tr].to(PDEV), Xte, y[te].to(PDEV), nc, TORCH_SEED + block + 1000 * s)[0]
                     for s in NOISE_SEEDS]
            out.setdefault("noise", {})[name] = {
                "boot_sd": float(np.std(bt, ddof=1)), "seed_sd": float(np.std(seeds, ddof=1)),
                "seed_values": [float(v) for v in seeds],
                "combined_sd": float(np.sqrt(np.var(bt, ddof=1) + np.var(seeds, ddof=1)))}
    out["chance"] = {"exact": 1.0 / T, "row": 1.0 / H, "col": 1.0 / W}
    del Xtr, Xte
    if PDEV == "cuda": torch.cuda.empty_cache()
    return out

def ridge_block(acts_train, acts_eval, layer):
    """APE axis protocol (metrics.position_probe): ridge on normalised row/col, 600/200/200 by image, train and
    evaluate on different RPI permutations, shuffled-label control."""
    if not RIDGE_OK: return {"error": "ridge import failed"}
    split = split_image_indices(int(acts_train.shape[0]), seed=0)
    r = evaluate_coordinate_probe(acts_train.numpy(), acts_eval.numpy(), split,
                                  shuffle_seed=1009 * (layer + 1), alpha_grid=DEFAULT_COORDINATE_PROBE_ALPHA_GRID)
    return {"row_r2": r["row"]["r2"], "col_r2": r["column"]["r2"],
            "row_null_r2": r["negative_control"]["row"]["r2"], "col_null_r2": r["negative_control"]["column"]["r2"],
            "row_alpha": r["row"]["alpha"], "col_alpha": r["column"]["alpha"]}

In [ ]:
# per-model setup
def load_model(key):
    spec = MODEL_SPECS[key]
    if TEST_MODE:
        tiny = dict(embed_dim=96, num_heads=6 if key != "naver" else 12, pretrained=False, dynamic_img_size=True)
        if key == "naver": tiny["embed_dim"] = 192
        if not spec["classifier"]: tiny["num_classes"] = 0
        else: tiny["num_classes"] = 10
        model = timm.create_model(spec["name"].split(".")[0], **tiny).to(DEVICE).eval()
        # random init uses layer-scale 1e-5, which makes every block a no-op and hides every bug;
        # open the residual branches and sharpen attention so rotations actually matter
        with torch.no_grad():
            for blk in model.blocks:
                for gname in ("gamma_1", "gamma_2"):
                    g_ = getattr(blk, gname, None)
                    if isinstance(g_, torch.Tensor): g_.fill_(1.0)
                q_ = getattr(blk.attn, "qkv", None)
                if q_ is not None: q_.weight.mul_(4.0)
                for pn in ("q_proj", "k_proj"):
                    m_ = getattr(blk.attn, pn, None)
                    if m_ is not None: m_.weight.mul_(4.0)
        cfg = timm.data.resolve_model_data_config(model); cfg["input_size"] = (3, INPUT_SIZE, INPUT_SIZE)
        return model, timm.data.create_transform(**cfg, is_training=False), "timm"
    return load_rope(model_name=spec["name"], device=DEVICE, half=False, input_size=INPUT_SIZE)

def needs_acc(key):
    return (MODEL_SPECS[key]["classifier"] and DS_ACC is not None and
            any(p in PARTS and key in PART_MODELS[p] for p in ("C", "D", "F")))

def setup_model(key):
    spec = MODEL_SPECS[key]
    print("\n" + "=" * 78 + f"\n{key}  ({spec['name']})\n" + "=" * 78, flush=True)
    model, processor, source = load_model(key)
    model.eval().float()
    rh, rh_src, nh_attr = attn_attrs(model, source)
    n_prefix = int(getattr(model, "num_prefix_tokens", 1))
    M = {"key": key, "model": model, "source": source, "processor": processor, "n_prefix": n_prefix,
         "hybrid": spec["hybrid"], "mixed": spec["mixed"], "classifier": spec["classifier"],
         "rotate_half": bool(rh), "rotate_half_source": rh_src, "n_heads": int(nh_attr), "cache": {}}
    M["cache"]["ssdc"] = build_cache(DS_SSDC, processor, source, N_SSDC)
    if needs_acc(key):
        M["cache"]["acc"] = build_cache(DS_ACC, processor, source, N_ACC)
    with torch.no_grad():
        g = get_patch_embed_conv(model, source)(M["cache"]["ssdc"][0][0][:1].to(DEVICE))
    Hg, Wg = int(g.shape[-2]), int(g.shape[-1]); M["grid"] = (Hg, Wg); T = Hg * Wg
    gate("forward grid is 14x14", key, (Hg, Wg) == (14, 14), f"{Hg}x{Wg}")
    fn, fn_name = final_norm_module(model); M["final_norm"] = fn; M["final_norm_name"] = fn_name
    print(f"  heads={M['n_heads']} prefix={n_prefix} rotate_half={rh} final readout via model.{fn_name}")
    A_ = ANCH.get(key)
    if A_ and not TEST_MODE:
        gate("num_heads matches W5", key, M["n_heads"] == A_["n_heads"], f"{M['n_heads']} vs {A_['n_heads']}")

    rname, rmod = find_rope(model); M["rope_name"], M["rope"] = rname, rmod
    if not spec["mixed"]:
        emb0 = rmod.get_embed(shape=(Hg, Wg))
        n_planes = int(emb0.shape[-1] // 4); M["n_planes"] = n_planes
        amap, rowp, colp, dec = derive_axis(emb0, n_planes, rh, Hg, Wg)
        M["row_planes"], M["col_planes"] = rowp, colp
        gate("rope rows equal patch tokens", key, int(emb0.shape[0]) == T, f"{emb0.shape[0]} vs {T}")
        gate("rotary table has no head dimension, so every head in a block uses the same frequencies", key,
             emb0.dim() == 2, f"shape {tuple(emb0.shape)}; {n_planes} planes per head")
        gate("every plane classifies decisively", key, dec, f"max ratio {max(a['ratio'] for a in amap):.1e}")
        gate("axis split is balanced", key, len(rowp) == len(colp) == n_planes // 2, f"{len(rowp)}/{len(colp)}")
        if A_ and not TEST_MODE:
            gate("row/col planes equal W5's runtime derivation", key,
                 rowp == A_["row_planes"] and colp == A_["col_planes"], f"row {rowp[:3]}..")
        M["ROWM"] = lambda e, _r=rowp, _n=n_planes, _h=rh: mask_planes(e, _r, _n, _h)
        M["COLM"] = lambda e, _c=colp, _n=n_planes, _h=rh: mask_planes(e, _c, _n, _h)

    # bootstrap machinery (paired: one resample table per model, used by every condition)
    n_img = sum(int(lb.numel()) for _, lb in M["cache"]["ssdc"])
    rng = np.random.default_rng(BOOT_SEED)
    idx = rng.integers(0, n_img, size=(BOOT_B, n_img))
    Wb = np.stack([np.bincount(r, minlength=n_img) for r in idx]).astype(np.float32)
    M["boot_w"] = torch.as_tensor(Wb, device=DEVICE)
    rk, iu = neg_dist_ranks(T)
    rk_t = torch.as_tensor(rk, dtype=torch.float32, device=DEVICE); M["rk_std"] = rk_t - rk_t.mean()
    M["iu_t"] = (torch.as_tensor(iu[0], device=DEVICE), torch.as_tensor(iu[1], device=DEVICE))

    # readout-location gate: FINAL = final norm of block-11 output
    if fn_name == "norm":
        got = {}
        h1 = get_vit_blocks(model, source)[-1].register_forward_hook(lambda m, i, o: got.update(a=fn(o.detach())))
        h2 = model.norm.register_forward_hook(lambda m, i, o: got.update(b=o.detach()))
        with torch.no_grad(): model(M["cache"]["ssdc"][0][0][:2].to(DEVICE))
        h1.remove(); h2.remove()
        gate("FINAL readout equals model.norm's own output", key, torch.equal(got["a"], got["b"]))
    else:
        gate("FINAL readout uses fc_norm per token (pooled model)", key, fn_name == "fc_norm", fn_name)

    # replay gates: cached path == the repo's streamed evaluate_ssdc / predict
    def streamed_ssdc(ctx):
        torch.manual_seed(TORCH_SEED); np.random.seed(TORCH_SEED % (2 ** 31))
        with ctx:
            s, _ = evaluate_ssdc(model, processor, DS_SSDC, source, RPI=True, number_images=N_SSDC,
                                 batch_size=BATCH_SIZE, metric=METRIC, half=False, num_workers=0)
        return s
    checks = [("baseline", NullCtx)]
    if not spec["mixed"]:
        checks.append(("Cref", lambda: RopeComposite(model, source, [(WINDOW, IDEN)])))
    for nm_, mk in checks:
        a = streamed_ssdc(mk()); b = measure(M, mk())["ssdc"][:N_BLOCKS]
        d = max(abs(x - y) for x, y in zip(a, b))
        gate(f"cached replay reproduces streamed evaluate_ssdc [{nm_}]", key, d <= EQUIV_TOL, f"max |delta| {d:.2e}")
    if "acc" in M["cache"]:
        n_chk = min(512, N_ACC)
        dl = prep_data(DS_ACC, processor, source, corruption_type=None, number_images=n_chk,
                       batch_size=BATCH_SIZE, half=False, num_workers=0)
        cs = []
        with torch.inference_mode():
            for im, lb in dl:
                cs.append(model(im.to(DEVICE)).argmax(-1).cpu() == torch.as_tensor(lb))
        cs = torch.cat(cs).numpy().astype(np.uint8)
        _, cc, _ = accuracy(M, NullCtx())
        gate(f"cached replay reproduces streamed top-1 on the first {n_chk} images", key,
             np.array_equal(cs, cc[:n_chk]), f"{int((cs != cc[:n_chk]).sum())} images differ")
    M["cache_digest"] = {k: cache_digest(v) for k, v in M["cache"].items()}
    return M

In [ ]:
# bookkeeping
RESULTS, BOOT, PERIMG = {}, {}, {}

def part_file(part, key): return OUT_DIR / f"w8_part{part}_{key}.json"

def save_part(part, key, R, complete):
    R["complete"] = complete; R["saved_utc"] = datetime.now(timezone.utc).isoformat()
    part_file(part, key).write_text(json.dumps(R, indent=1, default=float))
    if key in BOOT and BOOT[key]:
        np.savez_compressed(OUT_DIR / f"w8_boot_{key}.npz",
                            **{f"{c}__r{r}": v for c, d in BOOT[key].items() for r, v in d.items()})
    if key in PERIMG and PERIMG[key]:
        np.savez_compressed(OUT_DIR / f"w8_perimage_{key}.npz", **PERIMG[key])

def load_done(part, key):
    f = part_file(part, key)
    if RESUME and f.exists():
        R = json.loads(f.read_text())
        if R.get("complete"):
            print(f"  part {part} on {key}: already complete in {f.name}, loaded"); return R
    return None

def reload_side_files(key):
    for kind, store in (("boot", BOOT), ("perimage", PERIMG)):
        f = OUT_DIR / f"w8_{kind}_{key}.npz"
        if f.exists() and not store.get(key):
            z = np.load(f); d = {}
            if kind == "boot":
                for n in z.files:
                    c, r = n.rsplit("__r", 1); d.setdefault(c, {})[int(r)] = z[n]
            else:
                d = {n: z[n] for n in z.files}
            store[key] = d

def single_names(nh, blocks=POST): return [(k, h) for k in blocks for h in range(nh)]

def ranking_from_A(RA):
    c = RA["conditions"]; cref = c["Cref"]["ssdc"]
    rows = []
    for name, r in c.items():
        if name.startswith("s_b"):
            k, h = int(name.split("_")[1][1:]), int(name.split("_")[2][1:])
            rows.append({"block": k, "head": h, "cost12": cref[FINAL] - r["ssdc"][FINAL],
                         "cost11": cref[11] - r["ssdc"][11]})
    rows.sort(key=lambda d: (-d["cost12"], d["block"], d["head"]))
    return rows

def agate(name, scope, value, ref, tol):
    """Anchor gate: skipped (not failed) when there is no committed value or in test mode."""
    if ref is None or TEST_MODE:
        return gate(name, scope, True, "no anchor" if ref is None else "test mode", skip=True)
    d = abs(value - ref)
    return gate(name, scope, d <= tol, f"{value:.6f} vs {ref:.6f} (|delta| {d:.1e}, tol {tol:.0e})")

def max_dev(pairs):
    pairs = [(a, b) for a, b in pairs if a is not None and b is not None]
    return max((abs(a - b) for a, b in pairs), default=float("nan")), len(pairs)

## Part A - single (block, head) rebuilding sweep (SSDC)
Rotations off in blocks 0-5 (Cref), then one (block, head) in 6-11 also kept without rotations.
Every condition records SSDC entering every block and at the **output of block 11** (readout 12,
after the model's final LayerNorm), which is the first readout every block 6-11 can reach. The
twelve index groups and six whole-block knockouts are run alongside, so the old numbers can be
decomposed. All conditions share one paired image bootstrap; the top heads are re-run under two
more RPI seeds.

In [ ]:
def part_A(M):
    key, model, source, nh = M["key"], M["model"], M["source"], M["n_heads"]
    done = load_done("A", key)
    if done: return done
    print(f"\n--- part A: {key} ---", flush=True)
    R = {"model": key, "part": "A", "rule_single": RULE_SINGLE, "n_heads": nh, "window": WINDOW, "post": POST,
         "readouts": "0-11 = input of block b after its first LayerNorm (as W2-W7); 12 = output of block 11 "
                     f"through model.{M['final_norm_name']}", "conditions": {}, "seed_runs": {}}
    BOOT[key] = {k: v for k, v in BOOT.get(key, {}).items() if k.startswith("E_")}   # keep part E's resamples
    A_ = ANCH.get(key); t0 = time.time()

    def run(name, ctx, boot=True, headnorm=False, extra=None, quiet=True):
        r = measure(M, ctx, boot_readouts=(11, FINAL) if boot else (), headnorm=headnorm)
        rec = {"ssdc": r["ssdc"], "hooks": r["hooks"], "runtime_s": r["runtime_s"]}
        if headnorm: rec["head_norm"] = r["head_norm"]
        if extra: rec.update(extra)
        R["conditions"][name] = rec
        if boot: BOOT[key][name] = r["boot"]
        if not quiet:
            print(f"    {name:14s} b4={r['ssdc'][4]:+.6f} b11={r['ssdc'][11]:+.6f} out11={r['ssdc'][FINAL]:+.6f}"
                  f"  hooks={r['hooks']} {r['runtime_s']}s", flush=True)
        return rec

    base = run("baseline", NullCtx(), headnorm=True, quiet=False)
    flo = run("floor", RopeGlobal(model, IDEN), quiet=False)
    if M["hybrid"]:
        run("ape_floor", APEZero(model), quiet=False); run("both_floor", Both(model, IDEN), quiet=False)
    cref = run("Cref", RopeComposite(model, source, [(WINDOW, IDEN)]), headnorm=True, quiet=False)
    if A_:
        agate("baseline reproduces W5 at block 4", key, base["ssdc"][4], A_["baseline"][4], ANCHOR_TOL_SSDC)
        agate("floor reproduces W5 at block 4", key, flo["ssdc"][4], A_["floor"][4], ANCHOR_TOL_SSDC)
        agate("Cref reproduces W5 prefix_5 at block 11", key, cref["ssdc"][11], A_["Cref"][11], ANCHOR_TOL_SSDC)
    gate("floor equals baseline at block 0", key, abs(base["ssdc"][0] - flo["ssdc"][0]) <= EQUIV_TOL,
         f"{abs(base['ssdc'][0] - flo['ssdc'][0]):.1e}")
    rall = measure(M, RopeComposite(model, source, [(range(N_BLOCKS), per_head(range(nh), IDEN, nh))]))
    d = max(abs(a - b) for a, b in zip(rall["ssdc"], flo["ssdc"]))
    gate("per-head mask of ALL heads reproduces the global floor (all 13 readouts)", key, d <= EQUIV_TOL, f"{d:.1e}")
    # bootstrap check: one replicate with every image weighted once must give the repo SSDC
    saved_w = M["boot_w"]; M["boot_w"] = torch.ones_like(saved_w[:1])
    try:
        r1 = measure(M, RopeComposite(model, source, [(WINDOW, IDEN)]), boot_readouts=(11, FINAL))
    finally:
        M["boot_w"] = saved_w
    d = max(abs(float(r1["boot"][r][0]) - cref["ssdc"][r]) for r in (11, FINAL))
    gate("bootstrap SSDC with equal weights equals the repo SSDC (rank ties absent)", key, d <= 1e-5, f"{d:.1e}")

    # the twelve index groups (the paper's unit)
    for h in range(nh):
        run(f"grp_h{h}", win_plus(model, source, {k: [h] for k in POST}, nh), extra={"heads": [[k, h] for k in POST]})
    if A_:
        dev, n = max_dev([(R["conditions"][f"grp_h{h}"]["ssdc"][11], A_["grp_b11"].get(h)) for h in range(nh)])
        if TEST_MODE: gate("index groups reproduce W5 stage C at block 11", key, True, "test mode", skip=True)
        else: gate("index groups reproduce W5 stage C at block 11 (all heads)", key, dev <= ANCHOR_TOL_SSDC,
                   f"max |delta| {dev:.1e} over {n}")
    h0 = (A_["stageC"][0] if (A_ and not TEST_MODE) else 0)
    r_post = measure(M, RopeComposite(model, source, [(WINDOW, IDEN), (POST, per_head([h0], IDEN, nh))]))
    d = max(abs(a - b) for a, b in zip(r_post["ssdc"], R["conditions"][f"grp_h{h0}"]["ssdc"]))
    gate(f"six one-block segments equal W5's single POST segment for head {h0}", key, d <= EQUIV_TOL, f"{d:.1e}")

    # attention-level check: a single (block, head) mask changes that head's output and no other
    kb, hb = POST[0], h0
    grab = {}
    proj = get_block_attention(get_vit_blocks(model, source)[kb], source).proj
    hk = proj.register_forward_pre_hook(lambda m, args: grab.__setitem__("x", args[0].detach().clone()))
    xb = M["cache"]["ssdc"][0][0][:4].to(DEVICE)
    try:
        with torch.no_grad():
            with RopeComposite(model, source, [(WINDOW, IDEN)]): model(xb)
            ref_x = grab["x"]
            with win_plus(model, source, {kb: [hb]}, nh): model(xb)
            new_x = grab["x"]
    finally:
        hk.remove()
    hd = ref_x.shape[-1] // nh
    same_others = all(torch.equal(ref_x[..., h * hd:(h + 1) * hd], new_x[..., h * hd:(h + 1) * hd]) for h in range(nh) if h != hb)
    changed = not torch.equal(ref_x[..., hb * hd:(hb + 1) * hd], new_x[..., hb * hd:(hb + 1) * hd])
    gate(f"masking (block {kb}, head {hb}) changes that head's attention output and leaves the other {nh - 1} bit-identical",
         key, same_others and changed)

    # whole-block knockouts
    for k in POST:
        run(f"blk_b{k}", RopeComposite(model, source, [(WINDOW, IDEN), ([k], IDEN)]))
    r_ph = measure(M, win_plus(model, source, {POST[0]: list(range(nh))}, nh))
    d = max(abs(a - b) for a, b in zip(r_ph["ssdc"], R["conditions"][f"blk_b{POST[0]}"]["ssdc"]))
    gate(f"all heads of block {POST[0]} via per_head equals identity on that block", key, d <= EQUIV_TOL, f"{d:.1e}")

    # every single (block, head)
    worst, inert = 0.0, []
    for i, (k, h) in enumerate(single_names(nh)):
        rec = run(f"s_b{k}_h{h}", win_plus(model, source, {k: [h]}, nh), extra={"block": k, "head": h})
        worst = max(worst, max(abs(rec["ssdc"][j] - cref["ssdc"][j]) for j in range(0, k + 1)))
        if rec["hooks"] == 0: inert.append(f"s_b{k}_h{h}")
        if (i + 1) % nh == 0:
            print(f"    block {k} done ({i + 1}/{len(POST) * nh})  {time.time() - t0:.0f}s", flush=True)
            save_part("A", key, R, complete=False)
    gate("single-head masks leave every readout at or before their block untouched", key,
         worst <= UNTOUCHED_TOL, f"max |delta| {worst:.1e}")
    gate("every intervening condition registered hooks", key, not inert, ",".join(inert[:5]))

    rank = ranking_from_A(R)
    R["ranking_single"] = rank
    grp_cost = sorted([(cref["ssdc"][11] - R["conditions"][f"grp_h{h}"]["ssdc"][11], h) for h in range(nh)], reverse=True)
    R["ranking_group_b11"] = [h for _, h in grp_cost]
    print("  top singles (block, head, cost at out11, cost at b11):",
          [(d["block"], d["head"], round(d["cost12"], 4), round(d["cost11"], 4)) for d in rank[:8]])
    print("  index groups by b11 cost:", R["ranking_group_b11"][:6],
          (f" (W5 stage C: {A_['stageC'][:6]})" if A_ else ""))
    if A_ and not TEST_MODE:
        gate("index-group ranking matches W5 stage C (top 4)", key, R["ranking_group_b11"][:4] == A_["stageC"][:4],
             f"{R['ranking_group_b11'][:4]} vs {A_['stageC'][:4]}")

    # RPI-seed replicates for the top of the ranking
    top_s = [(d["block"], d["head"]) for d in rank[:A_SEED_TOPN]]
    top_g = R["ranking_group_b11"][:3]
    for sd in RPI_SEEDS_EXTRA:
        out = {"Cref": measure(M, RopeComposite(model, source, [(WINDOW, IDEN)]), seed=sd)["ssdc"]}
        for (k, h) in top_s:
            out[f"s_b{k}_h{h}"] = measure(M, win_plus(model, source, {k: [h]}, nh), seed=sd)["ssdc"]
        for h in top_g:
            out[f"grp_h{h}"] = measure(M, win_plus(model, source, {k: [h] for k in POST}, nh), seed=sd)["ssdc"]
        R["seed_runs"][str(sd)] = out
    R["runtime_s"] = round(time.time() - t0, 1)
    save_part("A", key, R, complete=True)
    gate("part A completed", key, True, f"{len(R['conditions'])} conditions, {R['runtime_s']}s")
    return R

## Part B - which axis does each single head rebuild? (probes)
One axis's planes off in blocks 0-5 (Dref, as W3-W5), then a head kept without rotations in 6-11.
The probe protocol is W5's (per-block seeding, 80/20 image split, peak test accuracy). References
and the named index groups are probed at blocks 7-11 (the Table 14 average) and at the output of
block 11; single heads at block 11 and the output of block 11. Probe noise is measured on the
references at the two single-head readouts.

In [ ]:
def part_B(M, RA):
    key, model, source, nh = M["key"], M["model"], M["source"], M["n_heads"]
    done = load_done("B", key)
    if done: return done
    print(f"\n--- part B: {key} ---", flush=True)
    H, W = M["grid"]; ROWM, COLM = M["ROWM"], M["COLM"]
    R = {"model": key, "part": "B", "heads_probed": ["row", "col"], "p_ref": P_REF, "p_single": P_SINGLE,
         "conditions": {}}
    A_ = ANCH.get(key); t0 = time.time()

    def run(name, ctx, readouts, noise_at=(), extra=None):
        r = measure(M, ctx, capture=readouts)
        probes = {str(b): probe_block(r["acts"][b], H, W, b, heads=("row", "col"), noise=(b in noise_at))
                  for b in readouts}
        del r["acts"]
        rec = {"ssdc": r["ssdc"], "probes": probes, "hooks": r["hooks"], "runtime_s": r["runtime_s"]}
        if extra: rec.update(extra)
        R["conditions"][name] = rec
        p = probes[str(readouts[-1])]
        print(f"    {name:20s} out11 row={p['row'] * 100:5.1f} col={p['col'] * 100:5.1f}  {time.time() - t0:.0f}s", flush=True)
        return rec

    run("baseline", NullCtx(), P_REF)
    run("Dref_row", RopeComposite(model, source, [(WINDOW, ROWM)]), P_REF, noise_at=(11, FINAL), extra={"axis": "row"})
    run("Dref_col", RopeComposite(model, source, [(WINDOW, COLM)]), P_REF, noise_at=(11, FINAL), extra={"axis": "col"})
    if A_:
        pairs = []
        for b in range(7, 12):
            for ax in ("row", "col"):
                pairs.append((R["conditions"]["baseline"]["probes"][str(b)][ax], A_["base_post"].get(b, {}).get(ax)))
                for ref in ("row", "col"):
                    pairs.append((R["conditions"][f"Dref_{ref}"]["probes"][str(b)][ax],
                                  A_["Dref_post"][ref].get(b, {}).get(ax)))
        dev, n = max_dev(pairs)
        if TEST_MODE: gate("references reproduce W5 probes at blocks 7-11", key, True, "test mode", skip=True)
        else: gate("references reproduce W5 probes at blocks 7-11", key, dev <= ANCHOR_TOL_PROBE,
                   f"max |delta| {dev * 100:.2f} pts over {n}")
        agate("Dref_row reproduces W5 SSDC at block 11", key, R["conditions"]["Dref_row"]["ssdc"][11], A_["Dref_row"][11], ANCHOR_TOL_SSDC)
        agate("Dref_col reproduces W5 SSDC at block 11", key, R["conditions"]["Dref_col"]["ssdc"][11], A_["Dref_col"][11], ANCHOR_TOL_SSDC)
    save_part("B", key, R, complete=False)

    named = B_NAMED_GROUPS.get(key, []) if not TEST_MODE else [0]
    for h in named:
        for ax, tf in (("row", ROWM), ("col", COLM)):
            run(f"grp_{ax}_h{h}", win_plus(model, source, {k: [h] for k in POST}, nh, window_tf=tf), P_REF,
                extra={"axis": ax, "heads": [[k, h] for k in POST]})
            ref = (A_ or {}).get("D_post", {}).get(h, {}).get(ax)
            if ref is not None and not TEST_MODE:
                dev, n = max_dev([(R["conditions"][f"grp_{ax}_h{h}"]["probes"][str(b)][a], ref.get(b, {}).get(a))
                                  for b in range(7, 12) for a in ("row", "col")])
                gate(f"index group h{h} ({ax} window) reproduces W5 stage D probes", key, dev <= ANCHOR_TOL_PROBE,
                     f"max |delta| {dev * 100:.2f} pts over {n}")
    save_part("B", key, R, complete=False)

    sel = [(k, h) for h in named for k in POST]
    if RA:
        sel += [(d["block"], d["head"]) for d in RA["ranking_single"][:B_TOPK_SINGLES]]
    sel = list(dict.fromkeys(sel))
    R["singles_selected"] = sel
    print(f"  {len(sel)} single heads x 2 windows")
    for i, (k, h) in enumerate(sel):
        for ax, tf in (("row", ROWM), ("col", COLM)):
            run(f"s_{ax}_b{k}_h{h}", win_plus(model, source, {k: [h]}, nh, window_tf=tf), P_SINGLE,
                extra={"axis": ax, "block": k, "head": h})
        if (i + 1) % 4 == 0: save_part("B", key, R, complete=False)
    R["runtime_s"] = round(time.time() - t0, 1)
    save_part("B", key, R, complete=True)
    gate("part B completed", key, True, f"{len(R['conditions'])} conditions, {R['runtime_s']}s")
    return R

## Part C - top-1 for every single head, every index group, and the hybrid
ILSVRC validation, first 5,000 images, clean order, per-image correctness kept for paired SEs.
References and index groups must reproduce W7 (Table 18); per-image agreement is checked where W7
kept per-image files. reg1-gap's index-group arms get per-image files again, which restores its
paired SEs. The hybrid gets the all-heads test that G.2 says was never run, plus top-1 with the
embedding removed (the symmetric number the mock review asked for).

In [ ]:
def part_C(M, RA):
    key, model, source, nh = M["key"], M["model"], M["source"], M["n_heads"]
    done = load_done("C", key)
    if done: return done
    if "acc" not in M["cache"]:
        gate("part C has its accuracy images", key, False, "no accuracy dataset"); return None
    print(f"\n--- part C: {key} ---", flush=True)
    R = {"model": key, "part": "C", "dataset": ACC_DATASET, "n_images": N_ACC, "conditions": {}, "w7_check": {}}
    PERIMG[key] = {k: v for k, v in PERIMG.get(key, {}).items() if k.startswith("D_")}  # keep part D's arms
    W7 = ACC_ANCH.get(key); W7i = W7_IMG.get(key, {})
    chk = W7 is not None and ACC_IS_ILSVRC and not TEST_MODE
    t0 = time.time()

    def run(name, ctx, w7name=None, extra=None, quiet=True):
        a, c, hooks = accuracy(M, ctx)
        rec = {"acc": a, "hooks": hooks}
        if extra: rec.update(extra)
        R["conditions"][name] = rec; PERIMG[key][name] = c
        if chk and w7name and w7name in W7:
            entry = {"w7": W7[w7name], "delta": a - W7[w7name]}
            if w7name in W7i: entry["perimage_agree"] = float((W7i[w7name] == c).mean())
            R["w7_check"][name] = entry
        if not quiet: print(f"    {name:22s} top-1 {a * 100:6.2f}  {time.time() - t0:.0f}s", flush=True)
        return a

    T = M["grid"][0] * M["grid"][1]
    perm = torch.randperm(T, generator=torch.Generator().manual_seed(SHUFFLE_PERM_SEED))
    run("baseline", NullCtx(), "baseline", quiet=False)
    run("floor", RopeGlobal(model, IDEN), "floor_identity", quiet=False)
    run("window", RopeComposite(model, source, [(WINDOW, IDEN)]), "window_identity", quiet=False)
    if M["hybrid"]:
        run("ape_off", APEZero(model), quiet=False)
        run("both_off", Both(model, IDEN), quiet=False)
        run("ape_off_window", _ApeAndWindow(model, source), quiet=False)
    for h in range(nh):
        run(f"win_h{h}", win_plus(model, source, {k: [h] for k in POST}, nh), f"win_h{h}")
        run(f"int_h{h}", heads_only(model, source, {k: [h] for k in POST}, nh), f"int_h{h}")
    r1 = (W7 or {}).get("_top", (ANCH.get(key) or {}).get("stageC", [0])[0])
    if TEST_MODE or r1 >= nh: r1 = 0
    run(f"win_h{r1}_shuffled", win_plus(model, source, {k: [r1] for k in POST}, nh, inner=shuffled_like(perm)),
        f"win_h{r1}_shuffled", quiet=False)
    if chk:
        dev = max(abs(v["delta"]) for v in R["w7_check"].values())
        ag = [v["perimage_agree"] for v in R["w7_check"].values() if "perimage_agree" in v]
        gate(f"references and index groups reproduce W7 top-1 ({len(R['w7_check'])} arms)", key,
             dev <= ANCHOR_TOL_ACC, f"max |delta| {dev * 100:.2f} pts")
        if ag:
            gate(f"per-image agreement with W7 ({len(ag)} arms with W7 per-image files)", key,
                 min(ag) >= PERIMG_AGREE_MIN, f"min {min(ag) * 100:.2f}%")
    else:
        gate("references reproduce W7 top-1", key, True,
             "no W7 anchor for this model" if W7 is None else "dataset/test mode", skip=True)
    save_part("C", key, R, complete=False)

    for k in POST:
        run(f"blk_b{k}", RopeComposite(model, source, [(WINDOW, IDEN), ([k], IDEN)]))
    for i, (k, h) in enumerate(single_names(nh)):
        run(f"s_b{k}_h{h}", win_plus(model, source, {k: [h]}, nh), extra={"block": k, "head": h})
        if (i + 1) % nh == 0:
            print(f"    singles: block {k} done  {time.time() - t0:.0f}s", flush=True)
            save_part("C", key, R, complete=False)
    if RA:
        rank = RA["ranking_single"]
        for d in rank[:C_INTACT_TOPN]:
            k, h = d["block"], d["head"]
            run(f"int_s_b{k}_h{h}", heads_only(model, source, {k: [h]}, nh), extra={"block": k, "head": h})
        k, h = rank[0]["block"], rank[0]["head"]
        run(f"s_b{k}_h{h}_shuffled", win_plus(model, source, {k: [h]}, nh, inner=shuffled_like(perm)),
            extra={"block": k, "head": h}, quiet=False)
    R["runtime_s"] = round(time.time() - t0, 1)
    save_part("C", key, R, complete=True)
    gate("part C completed", key, True, f"{len(R['conditions'])} arms, {R['runtime_s']}s")
    return R

class _ApeAndWindow:
    """Hybrid only: embedding removed and rotations off in blocks 0-5 (no position at all in the window)."""
    def __init__(s, model, source): s.a = APEZero(model); s.b = RopeComposite(model, source, [(WINDOW, IDEN)]); s.handles = []
    def __enter__(s): s.a.__enter__(); s.b.__enter__(); s.handles = [1] + s.b.handles; return s
    def __exit__(s, *e): s.b.__exit__(); s.a.__exit__(); s.handles = []

## Part D - is grouping by head index arbitrary?
Head index h in block 6 and head index h in block 7 share no parameters, so an index group is just
one particular "one head per block" set. We compare the index groups with (i) the best head in each
block, (ii) the six best single heads anywhere, and (iii) random one-head-per-block sets, on SSDC and
on top-1. Sets (i) and (ii) are chosen from part A's SSDC, so their top-1 is an out-of-sample check.

In [ ]:
def part_D(M, RA, RC):
    key, model, source, nh = M["key"], M["model"], M["source"], M["n_heads"]
    done = load_done("D", key)
    if done: return done
    if RA is None:
        gate("part D has part A's ranking", key, False, "run part A first"); return None
    print(f"\n--- part D: {key} ---", flush=True)
    rank = RA["ranking_single"]; cost = {(d["block"], d["head"]): d["cost12"] for d in rank}
    best = {k: [max(range(nh), key=lambda h: (cost[(k, h)], -h))] for k in POST}
    top6 = {}
    for d in rank[:6]: top6.setdefault(d["block"], []).append(d["head"])
    rng = np.random.default_rng(D_SEED + sum(map(ord, key)))
    sets = [[int(rng.integers(nh)) for _ in POST] for _ in range(D_RAND_SSDC)]
    R = {"model": key, "part": "D", "rule_best_per_block": RULE_BEST_PER_BLOCK, "rule_top6": RULE_TOP6,
         "best_per_block": {str(k): v for k, v in best.items()}, "top6": {str(k): v for k, v in top6.items()},
         "random_sets": sets, "ssdc": {}, "acc": {}}
    t0 = time.time()
    R["ssdc"]["best_per_block"] = measure(M, win_plus(model, source, best, nh))["ssdc"]
    R["ssdc"]["top6"] = measure(M, win_plus(model, source, top6, nh))["ssdc"]
    for i, s in enumerate(sets):
        R["ssdc"][f"rand{i:02d}"] = measure(M, win_plus(model, source, {k: [h] for k, h in zip(POST, s)}, nh))["ssdc"]
    print(f"    SSDC side done ({len(sets) + 2} sets) {time.time() - t0:.0f}s", flush=True)
    if "acc" in M["cache"]:
        PERIMG.setdefault(key, {})
        for name, hb in [("best_per_block", best), ("top6", top6)] + \
                        [(f"rand{i:02d}", {k: [h] for k, h in zip(POST, s)}) for i, s in enumerate(sets[:D_RAND_ACC])]:
            a, c, _ = accuracy(M, win_plus(model, source, hb, nh))
            R["acc"][name] = a; PERIMG[key][f"D_{name}"] = c
        print(f"    top-1 side done {time.time() - t0:.0f}s", flush=True)
    R["runtime_s"] = round(time.time() - t0, 1)
    save_part("D", key, R, complete=True)
    gate("part D completed", key, True, f"{R['runtime_s']}s")
    return R

## Part E - single-head carriers (blocks 0-3, read at block 4)
The carrying score in F.2/Table 15 removes one head index in every block of the intact model and
reads SSDC at block 4, which only blocks 0-3 can affect. Here each (block, head) in 0-3 is removed
alone, so the "one head carries 88%" style statements can name a block.

In [ ]:
def part_E(M, RA):
    key, model, source, nh = M["key"], M["model"], M["source"], M["n_heads"]
    done = load_done("E", key)
    if done: return done
    print(f"\n--- part E: {key} ---", flush=True)
    R = {"model": key, "part": "E", "readout": 4, "conditions": {}}
    BOOT.setdefault(key, {}); A_ = ANCH.get(key); t0 = time.time()
    def run(name, ctx, extra=None):
        r = measure(M, ctx, boot_readouts=(4,))
        rec = {"ssdc": r["ssdc"], "hooks": r["hooks"]}
        if extra: rec.update(extra)
        R["conditions"][name] = rec; BOOT[key][f"E_{name}"] = r["boot"]
        return rec
    base = run("baseline", NullCtx()); flo = run("floor", RopeGlobal(model, IDEN))
    for h in range(nh):
        run(f"grp_h{h}", RopeComposite(model, source, [(range(N_BLOCKS), per_head([h], IDEN, nh))]))
    if A_:
        dev, n = max_dev([(R["conditions"][f"grp_h{h}"]["ssdc"][4], A_["carrier_b4"].get(h)) for h in range(nh)])
        if TEST_MODE: gate("carrier index groups reproduce W5 stage A at block 4", key, True, "test mode", skip=True)
        else: gate("carrier index groups reproduce W5 stage A at block 4", key, dev <= ANCHOR_TOL_SSDC,
                   f"max |delta| {dev:.1e} over {n}")
    for k in EARLY:
        run(f"blk_b{k}", RopeComposite(model, source, [([k], IDEN)]))
    worst = 0.0
    for (k, h) in single_names(nh, EARLY):
        rec = run(f"s_b{k}_h{h}", heads_only(model, source, {k: [h]}, nh), extra={"block": k, "head": h})
        worst = max(worst, max(abs(rec["ssdc"][j] - base["ssdc"][j]) for j in range(0, k + 1)))
    gate("carrier single-head masks leave upstream readouts untouched", key, worst <= UNTOUCHED_TOL, f"{worst:.1e}")
    R["runtime_s"] = round(time.time() - t0, 1)
    save_part("E", key, R, complete=True)
    gate("part E completed", key, True, f"{len(R['conditions'])} conditions, {R['runtime_s']}s")
    return R

## Parts F and G - the axis claim without axial planes, and Table 9 with the APE metric
**G** re-measures Table 9 at block 4 (baseline, one axis's planes off in every block, all planes off)
with W5's 14-class probes, and adds the ridge-R² row/column probe that §4.2 uses for APE
(metrics.position_probe, 600/200/200 by image, train and test on different RPI permutations).
**F** does the same on the recipe-matched *mixed* RoPE ViT-B, where every plane's angle is
m_x·f_x + m_y·f_y, so no plane belongs to one axis. Removing one coordinate from every angle (f_y or
f_x set to zero) is still exact, so we can ask whether the other axis survives without a plane split.

In [ ]:
def probe_and_ridge(M, ctx_factory, layer=4, extra_readouts=()):
    H, W = M["grid"]
    readouts = (layer,) + tuple(extra_readouts)
    r1 = measure(M, ctx_factory(), capture=readouts)
    rec = {"ssdc": r1["ssdc"], "hooks": r1["hooks"], "probes": {}}
    rec["probes"][str(layer)] = probe_block(r1["acts"][layer], H, W, layer, heads=("exact", "row", "col"))
    for b in extra_readouts:
        rec["probes"][str(b)] = probe_block(r1["acts"][b], H, W, b, heads=("row", "col"))
    r2 = measure(M, ctx_factory(), capture=(layer,), seed=TORCH_SEED + 1)
    rec["ridge"] = ridge_block(r1["acts"][layer], r2["acts"][layer], layer)
    del r1, r2
    return rec

def part_G(M):
    key, model = M["key"], M["model"]
    done = load_done("G", key)
    if done: return done
    print(f"\n--- part G: {key} ---", flush=True)
    R = {"model": key, "part": "G", "layer": 4, "conditions": {}}; A_ = ANCH.get(key); t0 = time.time()
    conds = [("baseline", lambda: NullCtx()), ("axis_row", lambda: RopeGlobal(model, M["ROWM"])),
             ("axis_col", lambda: RopeGlobal(model, M["COLM"])), ("floor", lambda: RopeGlobal(model, IDEN))]
    pairs, spairs = [], []
    for name, cf in conds:
        rec = probe_and_ridge(M, cf); R["conditions"][name] = rec
        p = rec["probes"]["4"]; rg = rec["ridge"]
        print(f"    {name:9s} row {p['row'] * 100:5.1f} col {p['col'] * 100:5.1f} exact {p['exact'] * 100:5.1f} | "
              f"ridge R2 row {rg.get('row_r2', float('nan')):.3f} col {rg.get('col_r2', float('nan')):.3f}", flush=True)
        if A_ and A_["probe4"].get(name):
            pairs += [(p[a], A_["probe4"][name][a]) for a in ("exact", "row", "col")]
            spairs.append((rec["ssdc"][4], A_["ssdc4"][name]))
    if A_:
        dev, n = max_dev(pairs); sdev, sn = max_dev(spairs)
        if TEST_MODE:
            gate("Table 9 probes reproduced at block 4", key, True, "test mode", skip=True)
        else:
            gate("Table 9 probes reproduced at block 4", key, dev <= ANCHOR_TOL_PROBE, f"max |delta| {dev * 100:.2f} pts over {n}")
            gate("Table 9 SSDC reproduced at block 4", key, sdev <= ANCHOR_TOL_SSDC, f"max |delta| {sdev:.1e} over {sn}")
    R["runtime_s"] = round(time.time() - t0, 1)
    save_part("G", key, R, complete=True)
    gate("part G completed", key, True, f"{R['runtime_s']}s")
    return R

def part_F(M):
    key, model, source = M["key"], M["model"], M["source"]
    done = load_done("F", key)
    if done: return done
    print(f"\n--- part F: {key} ---", flush=True)
    rope = M["rope"]; H, W = M["grid"]
    R = {"model": key, "part": "F", "conditions": {}}; t0 = time.time()
    fr = rope.freqs.detach()
    gate("mixed rope exposes learned frequencies (2, depth, heads, planes)", key,
         fr.dim() == 4 and fr.shape[0] == 2 and fr.shape[1] == N_BLOCKS, str(tuple(fr.shape)))
    def emb_with(comps):
        with MixedFreqMask(rope, comps):
            return rope.get_embed(shape=(H, W)).detach().float()
    e_full = rope.get_embed(shape=(H, W)).detach().float()
    D2 = e_full.shape[-1] // 2
    def variation(e):
        s = e[..., :D2].reshape(e.shape[0], e.shape[1], H, W, -1)
        return (s - s.mean(dim=2, keepdim=True)).abs().mean().item(), (s - s.mean(dim=3, keepdim=True)).abs().mean().item()
    v_rows0, v_cols0 = variation(emb_with([1]))      # only component 0 left
    comp0_is_col = v_rows0 < 1e-6 * max(v_cols0, 1e-12)
    comp0_is_row = v_cols0 < 1e-6 * max(v_rows0, 1e-12)
    gate("component 0 depends on exactly one grid axis", key, comp0_is_col or comp0_is_row,
         f"variation across rows {v_rows0:.2e}, across columns {v_cols0:.2e}")
    ROWC, COLC = (1, 0) if comp0_is_col else (0, 1)
    R["row_component"], R["col_component"] = ROWC, COLC
    e_none = emb_with([0, 1])
    gate("both components zero gives exact identity rotations", key,
         torch.equal(e_none[..., :D2], torch.zeros_like(e_none[..., :D2])) and torch.equal(e_none[..., D2:], torch.ones_like(e_none[..., D2:])))
    gate("frequency mask restores the exact tensor on exit", key, torch.equal(rope.freqs.detach(), fr))
    fx, fy = fr[ROWC].abs().flatten(), fr[COLC].abs().flatten()
    ratio = (torch.minimum(fx, fy) / torch.maximum(fx, fy).clamp_min(1e-12))
    R["obliqueness"] = {"median_min_over_max": float(ratio.median()), "frac_above_0.5": float((ratio > 0.5).float().mean())}
    print(f"  row component = freqs[{ROWC}] | median |smaller|/|larger| per plane {R['obliqueness']['median_min_over_max']:.3f}")
    rng = np.random.default_rng(F_RAND_SEED)
    rand_planes = sorted(rng.choice(fr.shape[3], size=min(F_RAND_PLANES, fr.shape[3]), replace=False).tolist())
    R["random_planes"] = rand_planes
    glob_conds = [("baseline", lambda: NullCtx()), ("floor", lambda: MixedFreqMask(rope, [0, 1])),
                  ("row_off", lambda: MixedFreqMask(rope, [ROWC])), ("col_off", lambda: MixedFreqMask(rope, [COLC])),
                  ("rand_planes", lambda: MixedFreqMask(rope, [0, 1], planes=rand_planes))]
    for name, cf in glob_conds:
        rec = probe_and_ridge(M, cf, extra_readouts=(11, FINAL) if name == "baseline" else ())
        R["conditions"][name] = rec; p = rec["probes"]["4"]
        print(f"    {name:11s} b4 row {p['row'] * 100:5.1f} col {p['col'] * 100:5.1f} exact {p['exact'] * 100:5.1f} "
              f"| ridge row {rec['ridge'].get('row_r2', float('nan')):.3f} col {rec['ridge'].get('col_r2', float('nan')):.3f}", flush=True)
    win_conds = [("Cref", lambda: MixedFreqMask(rope, [0, 1], depths=WINDOW)),
                 ("win_row", lambda: MixedFreqMask(rope, [ROWC], depths=WINDOW)),
                 ("win_col", lambda: MixedFreqMask(rope, [COLC], depths=WINDOW))]
    for name, cf in win_conds:
        r = measure(M, cf(), capture=(11, FINAL))
        probes = {str(b): probe_block(r["acts"][b], H, W, b, heads=("row", "col")) for b in (11, FINAL)}
        R["conditions"][name] = {"ssdc": r["ssdc"], "probes": probes, "hooks": r["hooks"]}
        print(f"    {name:11s} out11 row {probes[str(FINAL)]['row'] * 100:5.1f} col {probes[str(FINAL)]['col'] * 100:5.1f}", flush=True)
    again = measure(M, NullCtx())["ssdc"]
    d = max(abs(a - b) for a, b in zip(again, R["conditions"]["baseline"]["ssdc"]))
    gate("model is unchanged after every frequency edit (baseline re-measured)", key, d <= EQUIV_TOL, f"{d:.1e}")
    if "acc" in M["cache"]:
        R["acc"] = {}
        for name, cf in glob_conds[:4] + win_conds:
            R["acc"][name] = accuracy(M, cf())[0]
        print("    top-1:", {k: round(v * 100, 2) for k, v in R["acc"].items()})
    R["runtime_s"] = round(time.time() - t0, 1)
    save_part("F", key, R, complete=True)
    gate("part F completed", key, True, f"{R['runtime_s']}s")
    return R

## Run
One model at a time; every part saves when it finishes. Re-running the notebook with
`W8_RESUME=1` loads finished (part, model) files and continues with the rest.

In [ ]:
T_RUN = time.time()
for key in MODELS_RUN:
    here = [p for p in PARTS if key in PART_MODELS.get(p, [])]
    if not here:
        continue
    if RESUME and all(part_file(p, key).exists() and json.loads(part_file(p, key).read_text()).get("complete") for p in here):
        print(f"\n{key}: parts {here} already complete, loading")
        for p in here: RESULTS[(key, p)] = json.loads(part_file(p, key).read_text())
        reload_side_files(key)
        continue
    reload_side_files(key)
    try:
        M = setup_model(key)
    except Exception as ex:
        traceback.print_exc(); gate("model setup completed", key, False, repr(ex)[:200]); continue
    RA = RESULTS.get((key, "A")) or (json.loads(part_file("A", key).read_text()) if part_file("A", key).exists() else None)
    if RA and not RA.get("complete"): RA = None
    for p in here:
        try:
            if p == "A": RA = part_A(M); RESULTS[(key, "A")] = RA
            elif p == "B": RESULTS[(key, "B")] = part_B(M, RA)
            elif p == "C": RESULTS[(key, "C")] = part_C(M, RA)
            elif p == "D": RESULTS[(key, "D")] = part_D(M, RA, RESULTS.get((key, "C")))
            elif p == "E": RESULTS[(key, "E")] = part_E(M, RA)
            elif p == "F": RESULTS[(key, "F")] = part_F(M)
            elif p == "G": RESULTS[(key, "G")] = part_G(M)
        except Exception as ex:
            traceback.print_exc(); gate(f"part {p} completed", key, False, repr(ex)[:200])
    del M
    if DEVICE == "cuda": torch.cuda.empty_cache()
print(f"\nall requested parts finished in {(time.time() - T_RUN) / 60:.1f} min")

## Analysis
Everything below reads the saved part files, so it can be re-run on its own after a restart.
Costs are "points lost against the reference": SSDC against Cref (or baseline for carriers), top-1
against the 0-5 window, probes against the one-axis window. Uncertainty: the paired image bootstrap
for SSDC, paired per-image SEs for top-1, and the W5 probe-noise recipe for probes.

In [ ]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

def M_is_hybrid(key): return bool(MODEL_SPECS[key]["hybrid"])
def RA_heads(key):
    ra = load_part(key, "A"); return ra["n_heads"] if ra else "?"

def load_part(key, p):
    if (key, p) in RESULTS and RESULTS[(key, p)]: return RESULTS[(key, p)]
    f = part_file(p, key)
    return json.loads(f.read_text()) if f.exists() else None

def perm_spearman(x, y, n_perm=10000, seed=0):
    x, y = np.asarray(x, float), np.asarray(y, float)
    rx, ry = rankdata(x), rankdata(y)
    rho = float(np.corrcoef(rx, ry)[0, 1])
    g = np.random.default_rng(seed)
    null = np.array([np.corrcoef(rx, g.permutation(ry))[0, 1] for _ in range(n_perm)])
    return rho, float((np.sum(np.abs(null) >= abs(rho)) + 1) / (n_perm + 1))

def partial_spearman(x, y, z, n_perm=10000, seed=0):
    def resid(a, b):
        A = np.vstack([b, np.ones_like(b)]).T; coef, *_ = np.linalg.lstsq(A, a, rcond=None); return a - A @ coef
    rx, ry, rz = rankdata(x), rankdata(y), rankdata(z)
    ex, ey = resid(rx, rz), resid(ry, rz)
    rho = float(np.corrcoef(ex, ey)[0, 1]); g = np.random.default_rng(seed)
    null = np.array([np.corrcoef(ex, g.permutation(ey))[0, 1] for _ in range(n_perm)])
    return rho, float((np.sum(np.abs(null) >= abs(rho)) + 1) / (n_perm + 1))

def paired(a, b):
    d = a.astype(float) - b.astype(float); return float(d.mean()), float(d.std(ddof=1) / np.sqrt(len(d)))

ANALYSIS, TABLES = {}, {k: [] for k in ("singles_ssdc", "singles_acc", "groups", "axis", "carriers", "random_sets")}
for key in MODELS_RUN:
    reload_side_files(key)
    out = {}
    RA, RB, RC, RD, RE = (load_part(key, p) for p in "ABCDE")
    # A
    if RA and RA.get("complete"):
        c = RA["conditions"]; cref, flo, base = c["Cref"]["ssdc"], c["floor"]["ssdc"], c["baseline"]["ssdc"]
        nh = RA["n_heads"]; rank = RA["ranking_single"]
        top = rank[0]
        a = {"cref_out11": cref[FINAL], "baseline_out11": base[FINAL], "floor_out11": flo[FINAL],
             "recovery_b11": (cref[11] - flo[11]) / (base[11] - flo[11]) if not M_is_hybrid(key) else None,
             "recovery_out11": (cref[FINAL] - flo[FINAL]) / (base[FINAL] - flo[FINAL]) if not M_is_hybrid(key) else None,
             "top_single": [top["block"], top["head"]], "top_single_cost_out11": top["cost12"],
             "top_single_share_of_recovery": top["cost12"] / (cref[FINAL] - flo[FINAL]) if not M_is_hybrid(key) else None}
        blk = {k: cref[FINAL] - c[f"blk_b{k}"]["ssdc"][FINAL] for k in POST}
        a["block_cost_out11"] = blk
        a["top_head_share_of_its_block"] = top["cost12"] / blk[top["block"]] if blk[top["block"]] > 0 else None
        cost12 = {(d["block"], d["head"]): d["cost12"] for d in rank}
        hn = c["Cref"].get("head_norm")
        groups = {}
        for h in range(nh):
            g = c[f"grp_h{h}"]["ssdc"]; members = [cost12[(k, h)] for k in POST]
            groups[h] = {"cost_b11": cref[11] - g[11], "cost_out11": cref[FINAL] - g[FINAL],
                         "sum_of_member_singles_out11": float(sum(members)),
                         "best_member": [POST[int(np.argmax(members))], h], "best_member_cost_out11": float(max(members))}
            groups[h]["best_member_share"] = (max(members) / groups[h]["cost_out11"]) if groups[h]["cost_out11"] > 1e-9 else None
            TABLES["groups"].append({"model": key, "head_index": h, **{k: v for k, v in groups[h].items() if k != "best_member"},
                                     "best_member_block": groups[h]["best_member"][0]})
        a["groups"] = {str(h): v for h, v in groups.items()}
        Bt = BOOT.get(key, {})
        if Bt.get("Cref") is not None and all(f"s_b{k}_h{h}" in Bt for (k, h) in cost12):
            keys_ = list(cost12)
            mat = np.stack([Bt["Cref"][FINAL] - Bt[f"s_b{k}_h{h}"][FINAL] for (k, h) in keys_])   # [n, B]
            winners = mat.argmax(0)
            i_top = keys_.index((top["block"], top["head"]))
            ranks = (-mat).argsort(0).argsort(0) + 1
            a["boot_p_top_single_stays_top"] = float((winners == i_top).mean())
            a["boot_rank_interval_top5"] = {f"b{d['block']}_h{d['head']}":
                [int(np.percentile(ranks[keys_.index((d["block"], d["head"]))], 2.5)),
                 int(np.percentile(ranks[keys_.index((d["block"], d["head"]))], 97.5))] for d in rank[:5]}
            i2 = keys_.index((rank[1]["block"], rank[1]["head"]))
            a["boot_top1_minus_top2_out11"] = [float(mat[i_top].mean() - mat[i2].mean()), float((mat[i_top] - mat[i2]).std(ddof=1))]
            gm = np.stack([Bt["Cref"][11] - Bt[f"grp_h{h}"][11] for h in range(nh)])
            gtop = int(np.argmax([groups[h]["cost_b11"] for h in range(nh)]))
            a["boot_p_top_index_group_stays_top_b11"] = float((gm.argmax(0) == gtop).mean())
            a["boot_top_index_group"] = gtop
            for i, (k, h) in enumerate(keys_):
                TABLES["singles_ssdc"].append({"model": key, "block": k, "head": h, "cost_out11": cost12[(k, h)],
                    "cost_b11": [d for d in rank if d["block"] == k and d["head"] == h][0]["cost11"],
                    "boot_sd": float(mat[i].std(ddof=1)), "rank": int([j for j, d in enumerate(rank) if d["block"] == k and d["head"] == h][0]) + 1,
                    "cref_head_norm": hn[k][h] if hn else None})
        sr = RA.get("seed_runs", {})
        if sr:
            names = [f"s_b{d['block']}_h{d['head']}" for d in rank[:A_SEED_TOPN]]
            stay = []
            for sd, runs in sr.items():
                cc = {n: runs["Cref"][FINAL] - runs[n][FINAL] for n in names if n in runs}
                stay.append(max(cc, key=cc.get) == names[0])
            a["rpi_seeds_top_single_still_top"] = f"{sum(stay)}/{len(stay)}"
        out["A"] = a
    # C
    if RC and RC.get("complete") and key in PERIMG:
        P = PERIMG[key]; cc = RC["conditions"]; win = cc["window"]["acc"]
        cA = {}
        if RA and RA.get("complete"):
            cA = {(d["block"], d["head"]): d["cost12"] for d in RA["ranking_single"]}
        nh = max(int(n.split("_h")[1].split("_")[0]) for n in cc if n.startswith("win_h")) + 1
        sing = []
        for n, r in cc.items():
            if n.startswith("s_b") and "shuffled" not in n:
                k, h = r["block"], r["head"]
                m_, se = paired(P["window"], P[n])
                sing.append({"model": key, "block": k, "head": h, "acc_cost": m_ * 100, "paired_se": se * 100,
                             "ssdc_cost_out11": cA.get((k, h))})
        TABLES["singles_acc"] += sing
        cst = {}
        for h in range(nh):
            m_, se = paired(P["window"], P[f"win_h{h}"]); cst[h] = (m_ * 100, se * 100)
        topg = (RA or {}).get("ranking_group_b11", [None])[0]
        if topg is None: topg = (ANCH.get(key) or {}).get("stageC", [0])[0]
        others = [h for h in range(nh) if h != topg]
        d_oth = (P["window"].astype(float) - P[f"win_h{topg}"]) - np.mean([P["window"].astype(float) - P[f"win_h{h}"] for h in others], 0)
        nxt = max(others, key=lambda h: cst[h][0])
        m2, se2 = paired(P[f"win_h{nxt}"], P[f"win_h{topg}"])
        c_ = {"window_top1": win * 100, "baseline_top1": cc["baseline"]["acc"] * 100, "floor_top1": cc["floor"]["acc"] * 100,
              "top_index_group": topg, "top_group_cost": cst[topg],
              "top_group_minus_mean_of_others": [float(d_oth.mean() * 100), float(d_oth.std(ddof=1) / np.sqrt(len(d_oth)) * 100)],
              "top_group_is_costliest": bool(all(cst[topg][0] >= cst[h][0] for h in others)),
              "next_group": nxt, "top_minus_next": [m2 * 100, se2 * 100]}
        if sing and cA:
            df = pd.DataFrame(sing)
            rho, p = perm_spearman(df["ssdc_cost_out11"], df["acc_cost"])
            c_["singles_spearman_ssdc_vs_top1"] = [rho, p, len(df)]
            hn = (RA["conditions"]["Cref"].get("head_norm") if RA else None)
            if hn:
                z = [hn[k][h] for k, h in zip(df["block"], df["head"])]
                c_["singles_partial_spearman_given_head_norm"] = list(partial_spearman(df["ssdc_cost_out11"], df["acc_cost"], z))
            tA = RA["ranking_single"][0]
            accs = df.sort_values("acc_cost", ascending=False).reset_index(drop=True)
            pos = accs.index[(accs["block"] == tA["block"]) & (accs["head"] == tA["head"])][0] + 1
            c_["top_ssdc_single"] = [tA["block"], tA["head"]]
            c_["top_ssdc_single_rank_in_top1_cost"] = [int(pos), len(df)]
            c_["chance_that_rank_is_1"] = 1 / len(df)
            row = df[(df["block"] == tA["block"]) & (df["head"] == tA["head"])].iloc[0]
            c_["top_ssdc_single_top1_cost"] = [float(row.acc_cost), float(row.paired_se)]
            gk = [h for h in range(nh)]
            if RA:
                gx = [RA["conditions"]["Cref"]["ssdc"][11] - RA["conditions"][f"grp_h{h}"]["ssdc"][11] for h in gk]
                c_["groups_spearman_ssdc_vs_top1"] = list(perm_spearman(gx, [cst[h][0] for h in gk])) + [len(gk)]
        c_["block_cost"] = {k: (win - cc[f"blk_b{k}"]["acc"]) * 100 for k in POST if f"blk_b{k}" in cc}
        if M_is_hybrid(key):
            c_["hybrid_refs"] = {n: cc[n]["acc"] * 100 for n in ("baseline", "floor", "ape_off", "both_off", "window", "ape_off_window") if n in cc}
        out["C"] = c_
    # D
    if RD and RA and RA.get("complete"):
        c = RA["conditions"]; cref = c["Cref"]["ssdc"]; nh = RA["n_heads"]
        d_ = {}
        for metric in ("ssdc", "acc"):
            src = RD.get(metric) or {}
            if not src: continue
            if metric == "ssdc":
                val = {n: cref[FINAL] - v[FINAL] for n, v in src.items()}
                idx = {h: cref[FINAL] - c[f"grp_h{h}"]["ssdc"][FINAL] for h in range(nh)}
            else:
                if not (RC and RC.get("complete")): continue
                w = RC["conditions"]["window"]["acc"]
                val = {n: (w - v) * 100 for n, v in src.items()}
                idx = {h: (w - RC["conditions"][f"win_h{h}"]["acc"]) * 100 for h in range(nh)}
            null = np.array([v for n, v in val.items() if n.startswith("rand")])
            g = np.random.default_rng(0)
            max12 = np.array([g.choice(null, size=nh, replace=True).max() for _ in range(10000)])
            topi = max(idx, key=idx.get)
            paper_g = RA.get("ranking_group_b11", [topi])[0]
            d_[metric] = {"null_mean": float(null.mean()), "null_p95": float(np.percentile(null, 95)), "n_null": len(null),
                          "paper_top_group": paper_g, "paper_top_group_cost": idx[paper_g],
                          "paper_top_group_percentile": float((null < idx[paper_g]).mean() * 100),
                          "index_group_percentiles": {str(h): float((null < v).mean() * 100) for h, v in idx.items()},
                          "top_index_group": topi, "top_index_group_cost": idx[topi],
                          "expected_max_of_n_random_sets": [float(max12.mean()), float(np.percentile(max12, 2.5)), float(np.percentile(max12, 97.5))],
                          "p_max_random_ge_top_group": float((max12 >= idx[topi]).mean()),
                          "best_per_block": val.get("best_per_block"), "top6": val.get("top6")}
            for n, v in val.items():
                TABLES["random_sets"].append({"model": key, "metric": metric, "set": n, "cost": v})
        out["D"] = d_
    # B
    if RB and RB.get("complete"):
        cb = RB["conditions"]
        def pr(name, r, ax): return cb[name]["probes"][str(r)][ax]
        bar = {}
        for r in (11, FINAL):
            sds = [cb["Dref_row"]["probes"][str(r)].get("noise", {}).get("row", {}).get("combined_sd"),
                   cb["Dref_col"]["probes"][str(r)].get("noise", {}).get("col", {}).get("combined_sd")]
            sds = [s for s in sds if s is not None]
            bar[r] = 3 * max(sds) * 100 if sds else float("nan")
        b_ = {"noise_bar_3sd_points": {str(k): v for k, v in bar.items()}, "singles": [], "groups": {}}
        for name, r in cb.items():
            if name.startswith("s_row_"):
                k, h = r["block"], r["head"]; colname = f"s_col_b{k}_h{h}"
                if colname not in cb: continue
                row = {"model": key, "block": k, "head": h}
                for ro in (11, FINAL):
                    rc_ = (pr("Dref_row", ro, "row") - pr(name, ro, "row")) * 100
                    cc_ = (pr("Dref_col", ro, "col") - pr(colname, ro, "col")) * 100
                    fav = "row" if (rc_ > bar[ro] and rc_ - cc_ > bar[ro]) else ("col" if (cc_ > bar[ro] and cc_ - rc_ > bar[ro]) else "none")
                    tag = "out11" if ro == FINAL else "b11"
                    row.update({f"row_cost_{tag}": rc_, f"col_cost_{tag}": cc_, f"favours_{tag}": fav})
                b_["singles"].append(row); TABLES["axis"].append(row)
        for name, r in cb.items():
            if name.startswith("grp_row_h"):
                h = int(name.split("_h")[1]); coln = f"grp_col_h{h}"
                if coln not in cb: continue
                b_["groups"][str(h)] = {
                    "row_cost_mean7_11": float(np.mean([(pr("Dref_row", b, "row") - pr(name, b, "row")) * 100 for b in range(7, 12)])),
                    "col_cost_mean7_11": float(np.mean([(pr("Dref_col", b, "col") - pr(coln, b, "col")) * 100 for b in range(7, 12)])),
                    "row_cost_out11": (pr("Dref_row", FINAL, "row") - pr(name, FINAL, "row")) * 100,
                    "col_cost_out11": (pr("Dref_col", FINAL, "col") - pr(coln, FINAL, "col")) * 100}
        def _ret(ax, ro):
            ch = cb["baseline"]["probes"][str(ro)]["chance"][ax]; den = pr("baseline", ro, ax) - ch
            return (pr(f"Dref_{ax}", ro, ax) - ch) / den if abs(den) > 1e-9 else float("nan")
        b_["recovery_of_removed_axis"] = {ax: {str(ro): _ret(ax, ro) for ro in (11, FINAL)} for ax in ("row", "col")}
        out["B"] = b_
    # E
    if RE and RE.get("complete"):
        ce = RE["conditions"]; b4, f4 = ce["baseline"]["ssdc"][4], ce["floor"]["ssdc"][4]; rng4 = b4 - f4
        loss = {(r["block"], r["head"]): (b4 - r["ssdc"][4]) / rng4 for n, r in ce.items() if n.startswith("s_b")}
        nh = max(h for _, h in loss) + 1
        topc = max(loss, key=loss.get)
        grp = {h: (b4 - ce[f"grp_h{h}"]["ssdc"][4]) / rng4 for h in range(nh)}
        gtop = max(grp, key=grp.get)
        e_ = {"top_single_carrier": list(topc), "top_single_share": loss[topc],
              "top_index_group": gtop, "top_index_group_share": grp[gtop],
              "best_member_of_top_group": max(((k, gtop) for k in EARLY), key=lambda kh: loss[kh]),
              "block_share": {k: (b4 - ce[f"blk_b{k}"]["ssdc"][4]) / rng4 for k in EARLY}}
        for (k, h), v in loss.items(): TABLES["carriers"].append({"model": key, "block": k, "head": h, "share_of_b4_structure": v})
        out["E"] = e_
    # F, G
    RF, RG = load_part(key, "F"), load_part(key, "G")
    for tag, Rx in (("F", RF), ("G", RG)):
        if Rx and Rx.get("complete"):
            out[tag] = {n: {"ssdc4": v["ssdc"][4], **{f"{a}@{b}": p[a] for b, p in v.get("probes", {}).items() for a in p if a in ("exact", "row", "col")},
                            **({f"ridge_{k}": v["ridge"][k] for k in ("row_r2", "col_r2", "row_null_r2", "col_null_r2")} if "row_r2" in v.get("ridge", {}) else {})}
                        for n, v in Rx["conditions"].items()}
            if tag == "F": out[tag]["_acc"] = Rx.get("acc"); out[tag]["_obliqueness"] = Rx.get("obliqueness")
    ANALYSIS[key] = out

In [ ]:
def show(title, obj):
    print("\n" + title); print("-" * len(title))
    if isinstance(obj, pd.DataFrame): print(obj.to_string(index=False) if len(obj) else "(empty)")
    else: print(json.dumps(obj, indent=1, default=float))

for key, out in ANALYSIS.items():
    print("\n" + "=" * 78 + f"\n{key}\n" + "=" * 78)
    if "A" in out:
        a = out["A"]
        print(f"A  top single head: block {a['top_single'][0]} head {a['top_single'][1]}  "
              f"cost at out11 {a['top_single_cost_out11']:.4f}"
              + (f" = {a['top_single_share_of_recovery'] * 100:.1f}% of the window's recovered structure" if a.get("top_single_share_of_recovery") else ""))
        if "boot_p_top_single_stays_top" in a:
            print(f"   bootstrap: stays top in {a['boot_p_top_single_stays_top'] * 100:.0f}% of resamples; "
                  f"top5 rank intervals {a['boot_rank_interval_top5']}; RPI seeds: {a.get('rpi_seeds_top_single_still_top')}")
            print(f"   top index group (b11): h{a['boot_top_index_group']} stays top in {a['boot_p_top_index_group_stays_top_b11'] * 100:.0f}% of resamples")
        if a.get("recovery_b11") is not None:
            print(f"   window recovery: {a['recovery_b11']:.3f} entering block 11 (paper), {a['recovery_out11']:.3f} at the output of block 11")
        print("   block costs at out11:", {k: round(v, 4) for k, v in a["block_cost_out11"].items()})
        gtab = pd.DataFrame([{"h": h, **{k: v for k, v in g.items() if k != "best_member"}, "best_block": g["best_member"][0]}
                             for h, g in a["groups"].items()]).sort_values("cost_b11", ascending=False).head(5)
        show("   index groups: cost and what their best member explains", gtab)
    if "B" in out:
        b = out["B"]
        print(f"B  probe noise bar (3 sd, points): {b['noise_bar_3sd_points']}")
        show("   single heads, axis costs (points)", pd.DataFrame(b["singles"]).drop(columns=["model"]).head(30))
        show("   index groups (Table 14 style, plus out11)", b["groups"])
    if "C" in out:
        c = out["C"]
        print(f"C  top-1: baseline {c['baseline_top1']:.2f} | window {c['window_top1']:.2f} | floor {c['floor_top1']:.2f}")
        print(f"   top index group h{c['top_index_group']}: cost {c['top_group_cost'][0]:.2f} +- {c['top_group_cost'][1]:.2f}; "
              f"above mean of others {c['top_group_minus_mean_of_others'][0]:.2f} +- {c['top_group_minus_mean_of_others'][1]:.2f}; "
              f"costliest: {c['top_group_is_costliest']}; vs next (h{c['next_group']}) {c['top_minus_next'][0]:.2f} +- {c['top_minus_next'][1]:.2f}")
        for k_ in ("top_ssdc_single", "top_ssdc_single_top1_cost", "top_ssdc_single_rank_in_top1_cost", "singles_spearman_ssdc_vs_top1",
                   "singles_partial_spearman_given_head_norm", "groups_spearman_ssdc_vs_top1", "hybrid_refs", "block_cost"):
            if k_ in c: print(f"   {k_}: {c[k_]}")
    if "D" in out:
        for metric, d in out["D"].items():
            print(f"D  [{metric}] random one-head-per-block sets: mean {d['null_mean']:.4f}, 95th pct {d['null_p95']:.4f} (n={d['n_null']}); "
                  f"paper's top group h{d['paper_top_group']} {d['paper_top_group_cost']:.4f} (percentile {d['paper_top_group_percentile']:.0f}); "
                  f"max index group h{d['top_index_group']} {d['top_index_group_cost']:.4f}; expected max of {RA_heads(key)} random sets "
                  f"{d['expected_max_of_n_random_sets'][0]:.4f}; P(max >= top group) {d['p_max_random_ge_top_group']:.2f}; "
                  f"best-per-block {d['best_per_block']}; top6 {d['top6']}")
    if "E" in out:
        e = out["E"]
        print(f"E  top single carrier block {e['top_single_carrier'][0]} head {e['top_single_carrier'][1]}: {e['top_single_share'] * 100:.1f}% of block-4 structure; "
              f"top index group h{e['top_index_group']} {e['top_index_group_share'] * 100:.1f}%, best member block {e['best_member_of_top_group'][0]}")
    for tag in ("G", "F"):
        if tag in out:
            show(f"{tag}  block-4 probes and ridge R2", pd.DataFrame({n: v for n, v in out[tag].items() if not n.startswith("_")}).T.reset_index().rename(columns={"index": "condition"}))
            if tag == "F": print("   F top-1:", out["F"].get("_acc"), "| obliqueness:", out["F"].get("_obliqueness"))

## Gate report and export

In [ ]:
# gates of every session; for a gate that ran more than once, the latest result counts
_all = []
for f in sorted(OUT_DIR.glob("w8_gates_session_*.json")):
    _all += json.loads(f.read_text())
_seen = {}
for g in _all: _seen[(g["scope"], g["gate"])] = g
gdf = pd.DataFrame(list(_seen.values()))
print(f"GATE REPORT  ({len(set(g['session'] for g in _all))} session(s), {len(_all)} gate calls, latest result per gate shown)\n")
print(gdf.to_string(index=False) if len(gdf) else "(none)")
n_fail = int((gdf.status == "FAIL").sum()) if len(gdf) else 0
n_skip = int((gdf.status == "SKIP").sum()) if len(gdf) else 0
print(f"\n{len(gdf)} gates: {int((gdf.status == 'PASS').sum())} pass, {n_fail} fail, {n_skip} skipped")
print("=" * 78)
print("ALL GATES PASSED" if n_fail == 0 else f"{n_fail} GATE(S) FAILED: report them before anyone uses the numbers")
for _, r in gdf[gdf.status == "FAIL"].iterrows():
    print(f"   - [{r['scope']}] {r['gate']}: {r['detail']}")
print("=" * 78)

export = {
    "notebook": "W8", "generated_utc": RUN_STAMP, "test_mode": TEST_MODE,
    "environment": {"torch": torch.__version__, "timm": timm.__version__, "device": DEVICE,
                    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu", "dtype": "float32",
                    "tf32_matmul": torch.backends.cuda.matmul.allow_tf32, "repo": str(repo)},
    "config": {"parts": PARTS, "models": MODELS_RUN, "n_ssdc": N_SSDC, "n_acc": N_ACC, "batch": BATCH_SIZE,
               "torch_seed": TORCH_SEED, "ssdc_dataset": SSDC_DATASET, "acc_dataset": ACC_DATASET,
               "window": WINDOW, "post": POST, "final_readout": "output of block 11 through the model's final LayerNorm",
               "boot": [BOOT_B, BOOT_SEED], "rpi_seeds_extra": RPI_SEEDS_EXTRA, "probe": [PROBE_LR, PROBE_PASSES, PROBE_BATCH, PROBE_TRAIN_FRAC],
               "rules": {"single": RULE_SINGLE, "best_per_block": RULE_BEST_PER_BLOCK, "top6": RULE_TOP6},
               "named_groups": B_NAMED_GROUPS, "anchor_tolerances": [ANCHOR_TOL_SSDC, ANCHOR_TOL_PROBE, ANCHOR_TOL_ACC]},
    "anchors_used": {"W5": {k: v["file"] for k, v in ANCH.items()}, "W7": {k: v.name for k, v in W7_FILES.items()}},
    "analysis": ANALYSIS, "gates": gdf.to_dict("records"),
    "gate_summary": {"total": len(gdf), "fail": n_fail, "skip": n_skip, "sessions": sorted(set(g["session"] for g in _all))},
}
(OUT_DIR / f"w8_all_{RUN_STAMP}.json").write_text(json.dumps(export, indent=1, default=float))
gdf.to_csv(OUT_DIR / "w8_gate_report.csv", index=False)
for name, rows in TABLES.items():
    if rows: pd.DataFrame(rows).to_csv(OUT_DIR / f"w8_{name}.csv", index=False)
archive = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
print("written:")
for p in sorted(OUT_DIR.iterdir()): print(f"  {p.name}  ({p.stat().st_size:,} bytes)")
print(f"\narchive: {archive} ({os.path.getsize(archive):,} bytes)")
try:
    from google.colab import files as _cf; _cf.download(archive)
except Exception:
    pass